# Brian LightGBM v6: Market + Sector Residual Alpha Audit

V6 answers two questions directly.

1. Does LightGBM beat a scaled linear ridge baseline on the same features and same target?
2. Does the LightGBM signal beat the defensive construction layer once portfolio effects are isolated?

The notebook intentionally cuts the earlier V6 draft back to one LightGBM ensemble, one ridge baseline, explicit market-neutralized sector residual target, explicit optimizer, alpha attribution, and hard self-checks.

Scope decisions:

- No regime-specialized model objects.
- No peer cohorts.
- No XGBoost or CatBoost.
- No SEC fundamentals.
- FRED non-revised daily macro features are lagged by 1 business day and cached.
- Static sector mapping is allowed, documented, and audited.

## V6.1 engineering decisions

Task 1 (decomposition): held the top-K selected universe constant and varied equal-weight, inverse-vol, model-rank, and cvxpy weights. Residual is printed as `alpha_decomposition_residual_pct_abs` and self-checked under 5%.
Task 2 (ridge portfolio): used a single final ridge model trained on all final training rows and replaced only the rank head. Justification: this isolates whether the LightGBM rank head adds useful nonlinear ordering.
Task 3 (rename + model_unused): selection can rank by `model_score` or inverse `expected_volatility`. Justification: `expected_volatility` is the portfolio-facing volatility signal and keeps the ablation schema identical.
Task 4 (leakage audit): sample size is at least 25 with regime, sector, short-history, train-start, and test-end coverage. FRED `_pit` columns are skipped because they are cached exogenous data already lagged by 1 business day.

Latest corrections:

- FRED daily non-revised series use a 1-business-day lag. The 30-day lag remains the default rule only for future revised or uncertain-release series.
- Target uses explicit market-neutralized sector residualization: `R_i - beta_i * R_mkt - (R_sector - beta_sector * R_mkt)`. This keeps the market residual and sector residual separate without subtracting market exposure twice.
- Optimizer sweep tunes `lambda_vol` and `lambda_turnover`; composite score weights stay fixed at 0.70/0.25/-0.15 to avoid expanding model-selection degrees of freedom in this diagnostic version.

V6.2 signal-isolation tests:

- `risk_residualized_score`: residualizes the final model score by date against volatility, beta, and sector exposures before portfolio construction.
- `no_risk_rank_head`: trains a rank-only LightGBM head after removing explicit volatility, beta, FF factor, and macro-risk columns. Magnitude and tail heads stay unchanged.
- These are ablations, not the default submission path. They test whether the model has signal beyond the low-volatility and beta-defense channel.

## External data sources

- Prices and benchmark: repository dataset preset `shared_set_1`, loaded through `portfolio_toolkit.load_prices`.
- Sector map: dataset spec sector metadata if available; otherwise a cached snapshot from Wikipedia's S&P 500 constituents table, https://en.wikipedia.org/wiki/List_of_S%26P_500_companies.
- Macro features: FRED CSV endpoints for DGS10, DGS2, VIXCLS, BAMLC0A4CBBB, and DTWEXBGS, https://fred.stlouisfed.org.
- Fama-French factors: Kenneth R. French Data Library daily Fama-French 5-factor file, https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html.
- Every external source is also recorded in `EXTERNAL_DATA_SOURCES` and copied into the saved metadata artifact.

In [1]:
# Beginner-safe dependency bootstrap.
#
# Run this first. It installs only packages missing from the current
# notebook kernel, using the exact Python executable that Jupyter is
# running. If this cell installs anything, restart the kernel once and
# continue from the top.

import importlib.util
import subprocess
import sys
from pathlib import Path


REQUIRED_PACKAGES = {
    "cvxpy": "cvxpy",
    "lightgbm": "lightgbm",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "pyarrow": "pyarrow",
    "lxml": "lxml",
    "bs4": "beautifulsoup4",
    "html5lib": "html5lib",
    "mlflow": "mlflow",
}


def _is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if _is_repo_root(candidate):
            return candidate.resolve()
    raise RuntimeError("Run this notebook from inside Portfolio-Optimization-Lib.")


print("Notebook Python:", sys.executable)
bootstrap_repo_root = _find_repo_root()
print("Repo root:", bootstrap_repo_root)

missing_packages = [
    pip_name
    for import_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])
    print("Install complete. Restart the kernel once, then rerun from the top.")
else:
    print("All required notebook packages are already available.")


Notebook Python: C:\Users\brixn\Documents\Portfolio-Optimization-Lib\venv312\Scripts\python.exe
Repo root: C:\Users\brixn\Documents\Portfolio-Optimization-Lib
All required notebook packages are already available.


In [2]:
import os
import sys
import json
import math
import itertools
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import cvxpy as cp
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)


def is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "portfolio_toolkit").exists()


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate
    raise RuntimeError("Run this notebook from inside Portfolio-Optimization-Lib.")


repo_root = find_repo_root()
os.chdir(repo_root)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from portfolio_toolkit import (
    PortfolioWeights,
    build_features,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    make_forward_return_target,
    start_run,
    validate_feature_frame,
    validate_prediction_frame,
    validate_weights_frame,
)

print("repo_root =", repo_root)
print("Imports successful.")


repo_root = C:\Users\brixn\Documents\Portfolio-Optimization-Lib
Imports successful.


In [24]:
# Configuration

DATASET_NAME = "shared_set_1"
MODEL_NAME = "Brian_lgbm_v6_market_sector_residual_alpha_audit"
STRATEGY_NAME = "brian_lgbm_v6_market_sector_residual_alpha_audit"

HORIZON = 10
N_RANK_BINS = 20
EMBARGO_DAYS = 20
COST_BPS = 10.0

TRAIN_START = pd.Timestamp("2014-01-02")
TRAIN_END = pd.Timestamp("2019-12-31")
VAL_START = pd.Timestamp("2020-01-02")
VAL_END = pd.Timestamp("2021-12-31")
TEST_START = pd.Timestamp("2022-01-03")
TEST_END = pd.Timestamp("2022-12-31")

SUBMISSION_MODE = False
RUN_MONTE_CARLO = bool(SUBMISSION_MODE)
MC_N_SIMS = 100 if SUBMISSION_MODE else 25
MC_BLOCK_SIZE = 10
RANDOM_SUBSET_SIMS = 25 if SUBMISSION_MODE else 5
RANDOM_SUBSET_FRAC = 0.70

ENSEMBLE_SEEDS = [17, 42, 101, 211]
RIDGE_ALPHA_GRID = [0.1, 1.0, 10.0, 100.0]
RANK_COMPONENT_WEIGHT = 0.70
MAGNITUDE_COMPONENT_WEIGHT = 0.25
TAIL_COMPONENT_WEIGHT = -0.15
COMPOSITE_SCORE_TUNING_NOTE = (
    "Composite score weights are held fixed at 0.70/0.25/-0.15. "
    "This version tunes optimizer penalties only so signal-blend tuning does not become another validation overfit channel."
)

RUN_MLFLOW = False
SAVE_LOCAL_ARTIFACTS = False

EXTERNAL_DATA_SOURCES = {
    "prices": {
        "name": "portfolio_toolkit shared_set_1",
        "access": "repository dataset preset via portfolio_toolkit.load_prices",
        "citation": "Portfolio-Optimization-Lib repository data preset shared_set_1",
        "notes": "Primary OHLCV price panel and benchmark used by the club toolkit.",
    },
    "sector_map": {
        "name": "S&P 500 sector map",
        "access": "dataset spec sector metadata when present, otherwise cached Wikipedia snapshot",
        "citation": "Wikipedia contributors, List of S&P 500 companies, https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "cache_path": "data/sector_map_snapshot.csv",
        "notes": "Static sector-as-of-snapshot approximation, not true historical point-in-time membership.",
    },
    "sector_manual_overrides": {
        "name": "Manual sector overrides for tickers missing from the static S&P 500 snapshot",
        "access": "hard-coded notebook overrides with cited profile source",
        "citation": "StockAnalysis, Coterra Energy (CTRA) Company Profile, https://stockanalysis.com/stocks/ctra/company/",
        "notes": "Used only when the repository universe contains a ticker absent from the cached static sector table.",
    },
    "fred_macro": {
        "name": "Federal Reserve Economic Data daily macro series",
        "series": ["DGS10", "DGS2", "VIXCLS", "BAMLC0A4CBBB", "DTWEXBGS"],
        "access": "FRED graph CSV endpoint",
        "citation": "Federal Reserve Bank of St. Louis, FRED, https://fred.stlouisfed.org",
        "cache_path": "data/fred_macro_pit_cache_lag1bd.parquet",
        "lag_rule": "1 business day for listed non-revised daily series; 30 calendar days default for future uncertain/revised series.",
    },
    "fama_french_5_factor": {
        "name": "Fama-French 5 Factors daily",
        "access": "Kenneth R. French Data Library ZIP CSV",
        "citation": "Kenneth R. French Data Library, Fama/French 5 Factors (2x3) daily, https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html",
        "download_url": "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip",
        "cache_path": "data/ff5_daily_cache.parquet",
    },
}

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
UNIVERSE_TICKERS = sorted(spec.tickers)
BENCHMARK = spec.benchmark_ticker.upper()

assert EMBARGO_DAYS >= 2 * HORIZON

print("Dataset:", DATASET_NAME, spec.name)
print("Universe size:", len(UNIVERSE_TICKERS))
print("Benchmark:", BENCHMARK)
print("Horizon:", HORIZON)
print("Embargo days:", EMBARGO_DAYS)
print("Submission mode:", SUBMISSION_MODE)


Dataset: shared_set_1 sp500_full_universe
Universe size: 503
Benchmark: SPY
Horizon: 10
Embargo days: 20
Submission mode: False


In [4]:
# Load prices and build weekly decision calendar.

prices = load_prices(DATASET_NAME, repo_root=repo_root)
prices["date"] = pd.to_datetime(prices["date"]).dt.tz_localize(None)
prices["ticker"] = prices["ticker"].astype(str).str.upper()
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)


def weekly_first_trading_day_calendar(prices_frame: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    all_dates = pd.DatetimeIndex(pd.to_datetime(prices_frame["date"].sort_values().unique()))
    candidate_dates = all_dates[(all_dates >= pd.Timestamp(start)) & (all_dates <= pd.Timestamp(end))]
    weekly = pd.DataFrame({"date": candidate_dates})
    weekly["week"] = weekly["date"].dt.to_period("W")
    execution_dates = pd.DatetimeIndex(weekly.groupby("week")["date"].first().to_numpy())
    rows = []
    for execution_date in execution_dates:
        pos = all_dates.searchsorted(execution_date, side="left")
        if pos > 0:
            rows.append({"signal_date": pd.Timestamp(all_dates[pos - 1]), "date": pd.Timestamp(execution_date)})
    return pd.DataFrame(rows)


decision_calendar = weekly_first_trading_day_calendar(prices, TRAIN_START, TEST_END)
price_wide = prices.pivot(index="date", columns="ticker", values="adj_close").sort_index()
returns_wide = price_wide.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)

print("Prices:", prices.shape)
print("Price range:", prices["date"].min().date(), "to", prices["date"].max().date())
print("Decision dates:", len(decision_calendar), decision_calendar["date"].min().date(), "to", decision_calendar["date"].max().date())
print(decision_calendar.head().to_string(index=False))


Prices: (1463605, 8)
Price range: 2014-01-02 to 2025-12-31
Decision dates: 469 2014-01-06 to 2022-12-27
signal_date       date
 2014-01-03 2014-01-06
 2014-01-10 2014-01-13
 2014-01-17 2014-01-21
 2014-01-24 2014-01-27
 2014-01-31 2014-02-03


In [5]:
# Static sector map audit.
#
# Preferred source is an existing toolkit/spec mapping. If absent, a one-time
# Wikipedia SP500 snapshot is cached under data/sector_map_snapshot.csv.
# This is a sector-as-of-snapshot approximation, not true historical PIT sector data.

SECTOR_MAP_PATH = repo_root / "data" / "sector_map_snapshot.csv"
SECTOR_MANUAL_OVERRIDES = {
    "CTRA": {
        "sector": "Energy",
        "source": "manual_override_stockanalysis_ctra_profile",
        "citation": "StockAnalysis, Coterra Energy (CTRA) Company Profile, https://stockanalysis.com/stocks/ctra/company/",
    },
}


def _sector_map_from_spec(spec_obj) -> dict[str, str] | None:
    for attr in ["sector_map", "sectors", "ticker_sectors", "gics_sector"]:
        value = getattr(spec_obj, attr, None)
        if isinstance(value, dict) and value:
            return {str(k).upper(): str(v) for k, v in value.items()}
    metadata = getattr(spec_obj, "metadata", None)
    if isinstance(metadata, dict):
        for key in ["sector_map", "sectors", "ticker_sectors", "gics_sector"]:
            value = metadata.get(key)
            if isinstance(value, dict) and value:
                return {str(k).upper(): str(v) for k, v in value.items()}
    return None


def load_or_fetch_sector_map() -> pd.DataFrame:
    existing = _sector_map_from_spec(spec)
    if existing:
        frame = pd.DataFrame({"ticker": list(existing.keys()), "sector": list(existing.values())})
        frame["source"] = "dataset_spec"
        return frame

    if SECTOR_MAP_PATH.exists():
        return pd.read_csv(SECTOR_MAP_PATH, comment="#")

    SECTOR_MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
    from urllib.request import Request, urlopen

    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    request = Request(url, headers={"User-Agent": "Mozilla/5.0 portfolio-research-notebook"})
    with urlopen(request, timeout=30) as response:
        html = response.read().decode("utf-8", errors="ignore")
    tables = pd.read_html(html)
    wiki = tables[0].rename(columns={"Symbol": "ticker", "GICS Sector": "sector"})
    frame = wiki.loc[:, ["ticker", "sector"]].copy()
    frame["ticker"] = frame["ticker"].astype(str).str.upper().str.replace(".", "-", regex=False)
    frame["sector"] = frame["sector"].astype(str)
    frame["source"] = f"wikipedia_sp500_snapshot_{pd.Timestamp.today().date()}"
    header = (
        "# Static SP500 sector map snapshot. Not true historical point-in-time sector membership.\n"
        f"# source={url}\n"
        f"# snapshot_date={pd.Timestamp.today().date()}\n"
    )
    SECTOR_MAP_PATH.write_text(header, encoding="utf-8")
    frame.to_csv(SECTOR_MAP_PATH, mode="a", index=False)
    return frame


sector_map = load_or_fetch_sector_map()
sector_map["ticker"] = sector_map["ticker"].astype(str).str.upper()
sector_map["sector"] = sector_map["sector"].astype(str)
override_rows = []
mapped_tickers = set(sector_map["ticker"])
for ticker, payload in SECTOR_MANUAL_OVERRIDES.items():
    if ticker in UNIVERSE_TICKERS and ticker not in mapped_tickers:
        override_rows.append({
            "ticker": ticker,
            "sector": payload["sector"],
            "source": payload["source"],
            "citation": payload["citation"],
        })
if override_rows:
    sector_map = pd.concat([sector_map, pd.DataFrame(override_rows)], ignore_index=True)
    print("Applied manual sector overrides:")
    print(pd.DataFrame(override_rows).to_string(index=False))
ticker_to_sector = dict(zip(sector_map["ticker"], sector_map["sector"]))

missing_sector = sorted(set(UNIVERSE_TICKERS) - set(ticker_to_sector))
if missing_sector:
    fallback_rows = pd.DataFrame({
        "ticker": missing_sector,
        "sector": "Unclassified",
        "source": "fallback_unclassified_missing_sector_mapping",
        "citation": "No external source available in cached sector map or manual overrides.",
    })
    sector_map = pd.concat([sector_map, fallback_rows], ignore_index=True)
    ticker_to_sector.update(dict(zip(fallback_rows["ticker"], fallback_rows["sector"])))
    print("Sector coverage note: missing sector mapping assigned to Unclassified:")
    print(fallback_rows.head(20).to_string(index=False))

sector_audit = []
for audit_date in [TRAIN_START, TRAIN_END, TEST_END]:
    available = prices.loc[(prices["date"] <= audit_date) & prices["ticker"].isin(UNIVERSE_TICKERS), "ticker"].drop_duplicates()
    counts = pd.Series([ticker_to_sector[t] for t in available if t in ticker_to_sector]).value_counts()
    sector_audit.append({"date": audit_date, **counts.to_dict()})
sector_audit_table = pd.DataFrame(sector_audit).fillna(0)

sector_changes = sector_map.groupby("ticker")["sector"].nunique()
sector_change_flags = sector_changes.loc[sector_changes > 1]

print("Sector source:", sector_map["source"].iloc[0] if "source" in sector_map else "unknown")
print("Sector counts at audit dates:")
print(sector_audit_table.to_string(index=False))
print("Tickers with sector changes in mapping:", sector_change_flags.to_dict())


Applied manual sector overrides:
ticker sector                                     source                                                                                             citation
  CTRA Energy manual_override_stockanalysis_ctra_profile StockAnalysis, Coterra Energy (CTRA) Company Profile, https://stockanalysis.com/stocks/ctra/company/
Sector source: wikipedia_sp500_snapshot_2026-05-20
Sector counts at audit dates:
      date  Industrials  Financials  Information Technology  Health Care  Consumer Discretionary  Consumer Staples  Utilities  Real Estate  Materials  Energy  Communication Services
2014-01-02           70          69                      61           55                      45                34         29           29         24      21                      20
2019-12-31           75          74                      69           56                      46                35         30           31         26      21                      23
2022-12-31           77 

In [6]:
# External data caches: FRED macro and Fama-French 5 factors.

DATA_DIR = repo_root / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_MANIFEST_PATH = DATA_DIR / "v6_external_data_sources.json"
FRED_CACHE_PATH = DATA_DIR / "fred_macro_pit_cache_lag1bd.parquet"
FF5_CACHE_PATH = DATA_DIR / "ff5_daily_cache.parquet"
SOURCE_MANIFEST_PATH.write_text(json.dumps(EXTERNAL_DATA_SOURCES, indent=2, sort_keys=True), encoding="utf-8")


def load_fred_macro_cache() -> pd.DataFrame:
    # TODO: upgrade to ALFRED vintage for v7.
    if FRED_CACHE_PATH.exists():
        return pd.read_parquet(FRED_CACHE_PATH)
    series_ids = ["DGS10", "DGS2", "VIXCLS", "BAMLC0A4CBBB", "DTWEXBGS"]
    frames = []
    for sid in series_ids:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}"
        raw = pd.read_csv(url)
        raw.columns = ["observation_date", sid]
        raw["observation_date"] = pd.to_datetime(raw["observation_date"])
        raw[sid] = pd.to_numeric(raw[sid].replace(".", np.nan), errors="coerce")
        frames.append(raw)
    macro = frames[0]
    for frame in frames[1:]:
        macro = macro.merge(frame, on="observation_date", how="outer")
    macro = macro.sort_values("observation_date").ffill()
    macro["yield_10y_2y_spread"] = macro["DGS10"] - macro["DGS2"]
    fred_default_revision_lag = pd.Timedelta(days=30)
    fred_known_daily_non_revised = {"DGS10", "DGS2", "VIXCLS", "BAMLC0A4CBBB", "DTWEXBGS"}
    if set(series_ids).issubset(fred_known_daily_non_revised):
        macro["date"] = macro["observation_date"] + pd.offsets.BDay(1)
        macro["fred_lag_rule"] = "one_business_day_non_revised_daily"
    else:
        macro["date"] = macro["observation_date"] + fred_default_revision_lag
        macro["fred_lag_rule"] = "thirty_calendar_days_default_revision_lag"
    rename = {
        "DGS10": "dgs10_pit",
        "DGS2": "dgs2_pit",
        "VIXCLS": "vixcls_pit",
        "BAMLC0A4CBBB": "bbb_oas_pit",
        "DTWEXBGS": "usd_index_pit",
        "yield_10y_2y_spread": "yield_10y_2y_spread_pit",
    }
    macro = macro.rename(columns=rename)
    keep = ["date", "fred_lag_rule", *rename.values()]
    macro = macro.loc[:, keep].dropna(subset=["date"]).sort_values("date")
    macro.to_parquet(FRED_CACHE_PATH, index=False)
    return macro


def load_ff5_cache() -> pd.DataFrame:
    if FF5_CACHE_PATH.exists():
        return pd.read_parquet(FF5_CACHE_PATH)
    import zipfile
    from io import BytesIO
    from urllib.request import urlopen

    url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
    with urlopen(url, timeout=30) as response:
        payload = response.read()
    with zipfile.ZipFile(BytesIO(payload)) as zf:
        name = zf.namelist()[0]
        text = zf.read(name).decode("latin1")
    lines = text.splitlines()
    start = next(i for i, line in enumerate(lines) if line.startswith(",Mkt-RF"))
    end = next(i for i in range(start + 1, len(lines)) if not lines[i].strip())
    ff = pd.read_csv(BytesIO("\n".join(lines[start:end]).encode()), index_col=0)
    ff.index = pd.to_datetime(ff.index.astype(str), format="%Y%m%d")
    ff = ff.rename_axis("date").reset_index()
    ff = ff.rename(columns={"Mkt-RF": "ff_mktrf", "SMB": "ff_smb", "HML": "ff_hml", "RMW": "ff_rmw", "CMA": "ff_cma"})
    for col in ["ff_mktrf", "ff_smb", "ff_hml", "ff_rmw", "ff_cma", "RF"]:
        ff[col] = pd.to_numeric(ff[col], errors="coerce") / 100.0
    ff.to_parquet(FF5_CACHE_PATH, index=False)
    return ff


macro_pit = load_fred_macro_cache()
ff5 = load_ff5_cache()

print("FRED PIT macro rows:", macro_pit.shape)
print("FRED PIT range:", macro_pit["date"].min().date(), "to", macro_pit["date"].max().date())
print("FRED lag rule:", macro_pit["fred_lag_rule"].dropna().iloc[-1] if "fred_lag_rule" in macro_pit else "unknown")
print("FF5 rows:", ff5.shape)
print("FF5 range:", ff5["date"].min().date(), "to", ff5["date"].max().date())
print("External data source manifest:", SOURCE_MANIFEST_PATH)
print(pd.DataFrame([
    {"source_key": key, "name": value["name"], "citation": value["citation"]}
    for key, value in EXTERNAL_DATA_SOURCES.items()
]).to_string(index=False))


FRED PIT macro rows: (16807, 8)
FRED PIT range: 1962-01-03 to 2026-05-20
FRED lag rule: one_business_day_non_revised_daily
FF5 rows: (15792, 7)
FF5 range: 1963-07-01 to 2026-03-31
External data source manifest: C:\Users\brixn\Documents\Portfolio-Optimization-Lib\data\v6_external_data_sources.json
             source_key                                                                         name                                                                                                                                     citation
                 prices                                               portfolio_toolkit shared_set_1                                                                               Portfolio-Optimization-Lib repository data preset shared_set_1
             sector_map                                                           S&P 500 sector map                                 Wikipedia contributors, List of S&P 500 companies, https://en.wikipedia.org/wiki/List

In [7]:
# Feature engineering.

TOOLKIT_FEATURES = [
    "return_1d", "return_5d", "return_10d", "return_20d", "return_60d",
    "momentum_5d", "momentum_10d", "momentum_20d", "momentum_60d", "momentum_120d",
    "vol_5d", "vol_20d", "vol_60d", "downside_vol_20d", "upside_vol_20d",
    "beta_20d_spy", "beta_60d_spy",
    "price_to_sma_20d", "price_to_sma_50d", "price_to_sma_200d",
    "macd_hist", "rsi_14", "bollinger_z_20d",
    "volume_zscore_20d", "volume_zscore_60d", "dollar_volume_ratio_20d",
    "intraday_range", "close_open_gap", "close_location_in_range",
    "distance_to_20d_high", "distance_to_20d_low", "distance_to_60d_high", "distance_to_60d_low",
    "excess_return_5d_vs_spy", "excess_return_20d_vs_spy", "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy", "skew_20d", "kurtosis_20d",
]


def _safe_zscore(series: pd.Series) -> pd.Series:
    std = series.std(ddof=0)
    if not np.isfinite(std) or std <= 1e-12:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std


def add_market_context(features: pd.DataFrame, prices_frame: pd.DataFrame) -> pd.DataFrame:
    spy = prices_frame.loc[prices_frame["ticker"] == BENCHMARK, ["date", "adj_close"]].drop_duplicates("date").sort_values("date")
    spy_ret = spy["adj_close"].pct_change(fill_method=None)
    context = spy.loc[:, ["date"]].copy()
    context["spy_return_20d"] = spy["adj_close"].pct_change(20, fill_method=None)
    context["spy_vol_20d"] = spy_ret.rolling(20, min_periods=20).std(ddof=0)
    context["spy_drawdown_60d"] = spy["adj_close"] / spy["adj_close"].rolling(60, min_periods=60).max() - 1.0
    return features.merge(context, on="date", how="left")


def add_rebound_features(features: pd.DataFrame, prices_frame: pd.DataFrame) -> pd.DataFrame:
    panel = prices_frame.sort_values(["ticker", "date"]).copy()
    spy_ret = (
        panel.loc[panel["ticker"] == BENCHMARK, ["date", "adj_close"]]
        .drop_duplicates("date")
        .sort_values("date")
        .assign(benchmark_return=lambda x: x["adj_close"].pct_change(fill_method=None))
        [["date", "benchmark_return"]]
    )
    panel["stock_return"] = panel.groupby("ticker", sort=False)["adj_close"].pct_change(fill_method=None)
    panel["prev_close"] = panel.groupby("ticker", sort=False)["close"].shift(1)
    panel = panel.merge(spy_ret, on="date", how="left")
    panel["down_day_excess"] = (panel["stock_return"] - panel["benchmark_return"]).where(panel["benchmark_return"] < 0.0)
    panel["down_day_count"] = panel["down_day_excess"].notna().astype(float)
    down_sum = panel.groupby("ticker", sort=False)["down_day_excess"].transform(lambda s: s.rolling(20, min_periods=1).sum())
    down_count = panel.groupby("ticker", sort=False)["down_day_count"].transform(lambda s: s.rolling(20, min_periods=1).sum())
    panel["relative_strength_down_days_20d"] = (down_sum / down_count).where(down_count >= 3)

    gap_mask = panel["open"] < panel["prev_close"] * 0.99
    denominator = (panel["prev_close"] - panel["open"]).replace(0.0, np.nan)
    panel["gap_absorption_raw"] = ((panel["close"] - panel["open"]) / denominator).where(gap_mask)
    panel["gap_down_absorption_20d"] = panel.groupby("ticker", sort=False)["gap_absorption_raw"].transform(
        lambda s: s.rolling(20, min_periods=1).mean()
    )

    out = features.merge(
        panel[["date", "ticker", "relative_strength_down_days_20d", "gap_down_absorption_20d"]],
        on=["date", "ticker"],
        how="left",
    )
    out["downside_vol_decay_60d"] = (
        out["downside_vol_20d"] - out.groupby("ticker", sort=False)["downside_vol_20d"].shift(40)
    ) / out.groupby("ticker", sort=False)["downside_vol_20d"].shift(40).replace(0.0, np.nan)
    beta_slope = out.groupby("ticker", sort=False)["beta_20d_spy"].transform(
        lambda s: s.rolling(20, min_periods=20).apply(lambda x: (x[-1] - x[0]) / 19.0, raw=True)
    )
    out["beta_stability_price_floor"] = beta_slope * (out["distance_to_60d_low"] > 0.05).astype(float)
    return out.replace([np.inf, -np.inf], np.nan)


def add_ff5_rolling_betas(features: pd.DataFrame, prices_frame: pd.DataFrame) -> pd.DataFrame:
    factor_cols = ["ff_mktrf", "ff_smb", "ff_hml", "ff_rmw", "ff_cma"]
    local_wide = prices_frame.loc[prices_frame["ticker"].isin(UNIVERSE_TICKERS)].pivot(index="date", columns="ticker", values="adj_close").sort_index()
    local_returns = local_wide.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    factor_frame = ff5.set_index("date").reindex(local_returns.index).ffill()
    stock_returns = local_returns.reindex(columns=UNIVERSE_TICKERS)
    rows = []
    for factor in factor_cols:
        f = factor_frame[factor]
        f_var = f.rolling(60, min_periods=40).var()
        beta = stock_returns.apply(lambda s: s.rolling(60, min_periods=40).cov(f) / f_var)
        long = beta.stack(dropna=False).rename(f"beta_60d_{factor}").reset_index().rename(columns={"level_1": "ticker"})
        rows.append(long)
    merged = rows[0]
    for frame in rows[1:]:
        merged = merged.merge(frame, on=["date", "ticker"], how="outer")
    return features.merge(merged, on=["date", "ticker"], how="left")


def add_cross_sectional_features(features: pd.DataFrame) -> pd.DataFrame:
    out = features.copy()
    grouped = out.groupby("date", sort=False)
    for column in [
        "momentum_20d", "momentum_60d", "momentum_120d", "vol_20d", "vol_60d",
        "downside_vol_20d", "beta_60d_spy", "excess_return_20d_vs_spy",
        "relative_strength_down_days_20d", "gap_down_absorption_20d",
        "downside_vol_decay_60d", "beta_stability_price_floor",
    ]:
        out[f"cs_rank_{column}"] = grouped[column].rank(pct=True)
        out[f"cs_z_{column}"] = grouped[column].transform(_safe_zscore)
    return out.replace([np.inf, -np.inf], np.nan)


def add_macro_features(features: pd.DataFrame) -> pd.DataFrame:
    left = features.sort_values("date")
    right = macro_pit.drop(columns=["fred_lag_rule"], errors="ignore").sort_values("date")
    merged = pd.merge_asof(left, right, on="date", direction="backward")
    return merged.sort_values(["ticker", "date"]).reset_index(drop=True)


def add_regime_columns(frame: pd.DataFrame, thresholds: dict[str, float]) -> pd.DataFrame:
    out = frame.copy()
    regime = pd.Series("calm", index=out.index, dtype="object")
    high_vol = out["spy_vol_20d"] >= thresholds["high_vol"]
    drawdown = out["spy_drawdown_60d"] <= thresholds["deep_drawdown"]
    rebound = out["spy_return_20d"] >= thresholds["strong_rebound"]
    regime.loc[high_vol] = "high_vol"
    regime.loc[drawdown] = "drawdown"
    regime.loc[rebound] = "rebound"
    out["regime"] = regime
    for name in ["calm", "drawdown", "high_vol", "rebound"]:
        out[f"regime_{name}"] = (out["regime"] == name).astype(float)
    out["stress_flag"] = out["regime"].isin(["high_vol", "drawdown"]).astype(float)
    return out


def build_model_features(prices_frame: pd.DataFrame) -> pd.DataFrame:
    base = build_features(prices_frame, feature_names=TOOLKIT_FEATURES)
    out = add_market_context(base, prices_frame)
    out = add_rebound_features(out, prices_frame)
    out = add_ff5_rolling_betas(out, prices_frame)
    out = add_macro_features(out)
    out = add_cross_sectional_features(out)
    return validate_feature_frame(out)


features = build_model_features(prices)
feature_columns = [c for c in features.columns if c not in {"date", "ticker", "regime"}]
print("Features built:", features.shape)
print("Feature count:", len(feature_columns))
print("Top missing rates:")
print(features[feature_columns].isna().mean().sort_values(ascending=False).head(15).to_string())


Features built: (1463605, 83)
Feature count: 81
Top missing rates:
bbb_oas_pit                        0.775512
gap_down_absorption_20d            0.345291
cs_rank_gap_down_absorption_20d    0.345291
cs_z_gap_down_absorption_20d       0.344978
price_to_sma_200d                  0.068422
cs_rank_momentum_120d              0.041272
momentum_120d                      0.041272
cs_rank_downside_vol_decay_60d     0.021103
downside_vol_decay_60d             0.021103
beta_60d_spy                       0.020652
return_60d                         0.020652
vol_60d                            0.020652
momentum_60d                       0.020652
excess_return_60d_vs_spy           0.020652
cs_rank_vol_60d                    0.020652


In [8]:
# Decision rows and explicit market-neutralized sector residual target.
#
# Question answered: can the label isolate stock-specific alpha without
# subtracting broad market exposure twice?
#
# Why the previous target failed: `R_i - beta_i * R_mkt - R_sector`
# subtracts the sector return including its own market beta. That double
# counts market exposure for high-beta sectors.
#
# Approaches considered:
# 1. Sector-only residual, `R_i - R_sector`. Simple, but leaves market
#    exposure inside the label.
# 2. Explicit market-neutralized sector residual,
#    `R_i - beta_i * R_mkt - (R_sector - beta_sector * R_mkt)`. More
#    explicit and matches the residual-alpha objective.
#
# Chosen approach: use the explicit market-neutralized sector residual
# because it preserves both market and sector neutralization without
# double-counting the market component.

FORWARD_RETURN_COL = f"forward_return_{HORIZON}d"
SECTOR_FORWARD_COL = f"sector_equal_weight_forward_return_{HORIZON}d"
MARKET_NEUTRAL_SECTOR_FORWARD_COL = f"market_neutral_sector_forward_return_{HORIZON}d"
TARGET_COL = f"target_market_neutral_sector_residual_alpha_{HORIZON}d"

features_for_signal = features.rename(columns={"date": "signal_date"})
feature_decisions = decision_calendar.merge(features_for_signal, on="signal_date", how="left")
feature_decisions = feature_decisions.loc[feature_decisions["ticker"].isin(UNIVERSE_TICKERS)].copy()
feature_decisions["sector"] = feature_decisions["ticker"].map(ticker_to_sector)

return_target = make_forward_return_target(prices, horizon=HORIZON)
return_target = return_target.loc[return_target["ticker"].isin(UNIVERSE_TICKERS + [BENCHMARK])].copy()
target_with_sector = return_target.loc[return_target["ticker"].isin(UNIVERSE_TICKERS)].copy()
target_with_sector["sector"] = target_with_sector["ticker"].map(ticker_to_sector)
sector_forward = (
    target_with_sector.groupby(["date", "sector"], sort=False)[FORWARD_RETURN_COL]
    .mean()
    .rename(SECTOR_FORWARD_COL)
    .reset_index()
)
benchmark_forward = (
    return_target.loc[return_target["ticker"] == BENCHMARK, ["date", FORWARD_RETURN_COL]]
    .rename(columns={FORWARD_RETURN_COL: "benchmark_forward_return"})
)

target_frame = (
    target_with_sector.merge(sector_forward, on=["date", "sector"], how="left")
    .merge(benchmark_forward, on="date", how="left")
)
model_frame = feature_decisions.merge(
    target_frame[["date", "ticker", FORWARD_RETURN_COL, SECTOR_FORWARD_COL, "benchmark_forward_return"]],
    on=["date", "ticker"],
    how="left",
)
model_frame["beta_for_target"] = model_frame["beta_60d_spy"].clip(lower=-1.0, upper=3.0).fillna(1.0)
model_frame["sector_beta_for_target"] = (
    model_frame.groupby(["date", "sector"], sort=False)["beta_for_target"]
    .transform("mean")
    .fillna(1.0)
)
model_frame[MARKET_NEUTRAL_SECTOR_FORWARD_COL] = (
    model_frame[SECTOR_FORWARD_COL]
    - model_frame["sector_beta_for_target"] * model_frame["benchmark_forward_return"]
)
model_frame[TARGET_COL] = (
    model_frame[FORWARD_RETURN_COL]
    - model_frame["beta_for_target"] * model_frame["benchmark_forward_return"]
    - model_frame[MARKET_NEUTRAL_SECTOR_FORWARD_COL]
)


def add_supervised_labels(frame: pd.DataFrame, target_col: str) -> pd.DataFrame:
    out = frame.copy()
    grouped = out.groupby("date", sort=False)[target_col]
    out["alpha_rank_pct"] = grouped.rank(pct=True)
    out["alpha_rank_label"] = np.floor(out["alpha_rank_pct"] * N_RANK_BINS).clip(0, N_RANK_BINS - 1).astype("Int64")
    out["alpha_zscore"] = grouped.transform(_safe_zscore)
    out["tail_loss_label"] = (out["alpha_rank_pct"] <= 0.20).astype("Int64")
    return out


model_frame = add_supervised_labels(model_frame, TARGET_COL)
required_targets = [
    FORWARD_RETURN_COL,
    SECTOR_FORWARD_COL,
    MARKET_NEUTRAL_SECTOR_FORWARD_COL,
    "benchmark_forward_return",
    TARGET_COL,
    "alpha_rank_label",
    "tail_loss_label",
]
model_frame = model_frame.dropna(subset=required_targets).reset_index(drop=True)

print("Decision feature rows:", feature_decisions.shape)
print("Labeled model rows:", model_frame.shape)
print("Execution dates:", model_frame["date"].nunique())
print("Label distribution:")
print(model_frame["alpha_rank_label"].value_counts().sort_index().to_string())
print("Timing audit:")
print(model_frame[[
    "signal_date", "date", "ticker", "sector",
    FORWARD_RETURN_COL, SECTOR_FORWARD_COL, MARKET_NEUTRAL_SECTOR_FORWARD_COL,
    "benchmark_forward_return", "beta_for_target", "sector_beta_for_target",
    TARGET_COL, "alpha_rank_label",
]].head().to_string(index=False))


Decision feature rows: (224439, 85)
Labeled model rows: (224439, 96)
Execution dates: 469
Label distribution:
alpha_rank_label
0     10947
1     11221
2     11270
3     11206
4     11153
5     11276
6     11210
7     11242
8     11238
9     11077
10    11359
11    11190
12    11290
13    11205
14    11123
15    11263
16    11254
17    11265
18    11226
19    11424
Timing audit:
signal_date       date ticker                 sector  forward_return_10d  sector_equal_weight_forward_return_10d  market_neutral_sector_forward_return_10d  benchmark_forward_return  beta_for_target  sector_beta_for_target  target_market_neutral_sector_residual_alpha_10d  alpha_rank_label
 2014-01-03 2014-01-06      A            Health Care            0.074329                                0.048530                                  0.038550                   0.00998              1.0                     1.0                                         0.025799                15
 2014-01-03 2014-01-06   AAPL Information

In [9]:
# Folds, regimes, feature columns, and leakage audit.
#
# Task 4 question: do live-frame features match features rebuilt from prices
# truncated at signal_date across regimes, sectors, short-history names, and
# edge dates?
#
# Why the old implementation failed: three random rows only catch broad
# systematic leakage and miss regime- or ticker-specific timing bugs.
#
# Approaches considered:
# 1. Rebuild all features for 25 independent truncation dates. Strongest
#    coverage but expensive.
# 2. Cache rebuilds by signal_date and choose 25 rows across fewer dates.
#    Same row-level coverage with lower runtime.
# 3. Recompute only hand-written features directly. Fastest but risks
#    diverging from the production feature builder.
#
# Chosen approach: cache full feature rebuilds by signal_date, then compare
# focused audit columns. This uses the production feature builder while
# controlling runtime. FRED _pit columns are skipped because they are
# exogenous cached data already lagged by 1 business day.
#
# What could break this: build_model_features must remain deterministic.
# Any future randomness or API refresh inside feature construction could
# create false leakage failures.

final_train_cutoff = TEST_START - pd.Timedelta(days=EMBARGO_DAYS)
pre_regime_train = model_frame.loc[(model_frame["date"] >= TRAIN_START) & (model_frame["date"] <= final_train_cutoff)].copy()
date_frame = pre_regime_train.drop_duplicates("date")
regime_thresholds = {
    "high_vol": float(date_frame["spy_vol_20d"].quantile(0.75)),
    "deep_drawdown": float(date_frame["spy_drawdown_60d"].quantile(0.25)),
    "strong_rebound": float(date_frame["spy_return_20d"].quantile(0.75)),
}
model_frame = add_regime_columns(model_frame, regime_thresholds)
feature_decisions = add_regime_columns(feature_decisions, regime_thresholds)


def make_walk_forward_folds(frame: pd.DataFrame) -> list[dict[str, object]]:
    folds = []
    for year in [2018, 2019, 2020, 2021]:
        val_start = pd.Timestamp(f"{year}-01-01")
        val_end = pd.Timestamp(f"{year}-12-31")
        purge_cutoff = val_start - pd.Timedelta(days=EMBARGO_DAYS)
        folds.append({
            "name": f"wf_{year}",
            "train": frame.loc[(frame["date"] >= TRAIN_START) & (frame["date"] <= purge_cutoff)].copy(),
            "val": frame.loc[(frame["date"] >= val_start) & (frame["date"] <= val_end)].copy(),
            "purge_cutoff": purge_cutoff,
        })
    return folds


folds = make_walk_forward_folds(model_frame)
final_train = model_frame.loc[(model_frame["date"] >= TRAIN_START) & (model_frame["date"] <= final_train_cutoff)].copy()
test_features = feature_decisions.loc[(feature_decisions["date"] >= TEST_START) & (feature_decisions["date"] <= TEST_END)].copy()
test_labeled = model_frame.loc[(model_frame["date"] >= TEST_START) & (model_frame["date"] <= TEST_END)].copy()

NON_FEATURE_COLUMNS = {
    "date", "signal_date", "ticker", "sector", "regime",
    FORWARD_RETURN_COL, SECTOR_FORWARD_COL, MARKET_NEUTRAL_SECTOR_FORWARD_COL,
    "benchmark_forward_return", "beta_for_target", "sector_beta_for_target", TARGET_COL,
    "alpha_rank_pct", "alpha_rank_label", "alpha_zscore", "tail_loss_label",
}
candidate_feature_cols = [
    c for c in final_train.columns
    if c not in NON_FEATURE_COLUMNS and pd.api.types.is_numeric_dtype(final_train[c])
]
training_frames_for_feature_audit = {"final_train": final_train}
training_frames_for_feature_audit.update({fold["name"]: fold["train"] for fold in folds})
dropped_feature_rows = []
for column in candidate_feature_cols:
    empty_in = [
        name
        for name, frame in training_frames_for_feature_audit.items()
        if column in frame.columns and frame[column].notna().sum() == 0
    ]
    if empty_in:
        dropped_feature_rows.append({"feature": column, "empty_in": ",".join(empty_in)})
DROPPED_UNUSABLE_FEATURES = pd.DataFrame(dropped_feature_rows)
dropped_feature_set = set(DROPPED_UNUSABLE_FEATURES["feature"]) if not DROPPED_UNUSABLE_FEATURES.empty else set()
FEATURE_COLS = [column for column in candidate_feature_cols if column not in dropped_feature_set]
if DROPPED_UNUSABLE_FEATURES.empty:
    print("No all-empty training feature columns were dropped.")
else:
    print("Dropped all-empty training feature columns:")
    print(DROPPED_UNUSABLE_FEATURES.to_string(index=False))
assert FEATURE_COLS, "No usable feature columns remain after all-empty feature filtering."
NO_RISK_FEATURE_EXCLUSION_TOKENS = [
    "beta", "ff_", "vix", "dgs", "yield", "oas",
    "vol_", "_vol", "volatility", "drawdown", "skew", "kurtosis",
    "intraday_range", "distance_to_", "spy_vol",
]
NO_RISK_FEATURE_COLS = [
    column for column in FEATURE_COLS
    if not any(token in column.lower() for token in NO_RISK_FEATURE_EXCLUSION_TOKENS)
]
NO_RISK_DROPPED_FEATURES = [column for column in FEATURE_COLS if column not in NO_RISK_FEATURE_COLS]
assert len(NO_RISK_FEATURE_COLS) >= 10, "No-risk rank head has too few features left."
CATEGORICAL_FEATURES = []

for fold in folds:
    print(
        fold["name"],
        "train_rows=", len(fold["train"]),
        "val_rows=", len(fold["val"]),
        "train_dates=", fold["train"]["date"].nunique(),
        "val_dates=", fold["val"]["date"].nunique(),
        "purge_cutoff=", fold["purge_cutoff"].date(),
    )
print("Regime thresholds:")
print(pd.Series(regime_thresholds).to_string())
print("Final train rows:", final_train.shape, "through", final_train["date"].max().date())
print("Test scoring rows:", test_features.shape)
print("Feature count:", len(FEATURE_COLS))
print("No-risk rank head feature count:", len(NO_RISK_FEATURE_COLS))
print("No-risk rank head dropped risk-feature count:", len(NO_RISK_DROPPED_FEATURES))
print("No-risk rank head kept features sample:", NO_RISK_FEATURE_COLS[:20])

def _first_row(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.head(1).copy()


audit_pool = feature_decisions.dropna(subset=["signal_date", "ticker", "sector", "regime"]).copy()
audit_parts = []
for regime_name, group in audit_pool.groupby("regime", sort=True):
    audit_parts.append(_first_row(group.sort_values(["date", "ticker"])))
short_history = audit_pool.loc[audit_pool["price_to_sma_200d"].isna()].sort_values(["date", "ticker"])
if not short_history.empty:
    audit_parts.append(_first_row(short_history))
start_near = audit_pool.loc[audit_pool["date"] <= TRAIN_START + pd.Timedelta(days=120)].sort_values(["date", "ticker"])
end_near = audit_pool.loc[audit_pool["date"] >= TEST_END - pd.Timedelta(days=90)].sort_values(["date", "ticker"])
if not start_near.empty:
    audit_parts.append(_first_row(start_near))
if not end_near.empty:
    audit_parts.append(_first_row(end_near))
sector_reference_date = audit_pool["date"].max()
sector_pool = audit_pool.loc[audit_pool["date"] == sector_reference_date]
if sector_pool.empty:
    sector_pool = audit_pool
for sector_name, group in sector_pool.groupby("sector", sort=True):
    audit_parts.append(_first_row(group.sort_values("ticker")))

audit_sample = (
    pd.concat(audit_parts, ignore_index=True)
    .drop_duplicates(["signal_date", "ticker"])
    .reset_index(drop=True)
)
preferred_signal_dates = audit_sample["signal_date"].drop_duplicates().tolist()
filler = audit_pool.loc[audit_pool["signal_date"].isin(preferred_signal_dates)].copy()
if len(audit_sample) < 25 and not filler.empty:
    needed = 25 - len(audit_sample)
    audit_sample = pd.concat(
        [audit_sample, filler.sample(min(needed, len(filler)), random_state=2029)],
        ignore_index=True,
    ).drop_duplicates(["signal_date", "ticker"]).reset_index(drop=True)
if len(audit_sample) < 25:
    needed = 25 - len(audit_sample)
    audit_sample = pd.concat(
        [audit_sample, audit_pool.sample(min(needed, len(audit_pool)), random_state=2030)],
        ignore_index=True,
    ).drop_duplicates(["signal_date", "ticker"]).reset_index(drop=True)
audit_sample = audit_sample.head(max(25, min(len(audit_sample), 40))).copy()
LEAKAGE_AUDIT_SAMPLE_SIZE = int(len(audit_sample))

audit_cols = [
    "return_20d", "momentum_60d", "vol_20d",
    "relative_strength_down_days_20d", "gap_down_absorption_20d",
    "downside_vol_decay_60d", "beta_stability_price_floor",
    "beta_60d_ff_mktrf", "beta_60d_ff_hml",
]
audit_cols = [col for col in audit_cols if col in features.columns]
rebuilt_by_signal_date = {}
for signal_date in pd.DatetimeIndex(audit_sample["signal_date"].drop_duplicates()).sort_values():
    truncated = prices.loc[prices["date"] <= signal_date].copy()
    rebuilt_by_signal_date[pd.Timestamp(signal_date)] = build_model_features(truncated)

for _, row in audit_sample.iterrows():
    rebuilt = rebuilt_by_signal_date[pd.Timestamp(row["signal_date"])]
    original = features.loc[(features["date"] == row["signal_date"]) & (features["ticker"] == row["ticker"])]
    recomputed = rebuilt.loc[(rebuilt["date"] == row["signal_date"]) & (rebuilt["ticker"] == row["ticker"])]
    assert len(original) == 1 and len(recomputed) == 1
    for col in audit_cols:
        a = float(original[col].iloc[0]) if pd.notna(original[col].iloc[0]) else np.nan
        b = float(recomputed[col].iloc[0]) if pd.notna(recomputed[col].iloc[0]) else np.nan
        assert (pd.isna(a) and pd.isna(b)) or np.isclose(a, b, rtol=1e-9, atol=1e-12), (row["ticker"], row["signal_date"], col, a, b)
assert LEAKAGE_AUDIT_SAMPLE_SIZE >= 25
LEAKAGE_AUDIT_PASSED = True
print("Leakage audit passed for sampled signal-date feature rebuilds.")
print("Leakage audit sample size:", LEAKAGE_AUDIT_SAMPLE_SIZE)
print("Leakage audit regimes:", sorted(audit_sample["regime"].dropna().unique().tolist()))
print("Leakage audit sectors:", sorted(audit_sample["sector"].dropna().unique().tolist()))
print("Leakage audit short-history rows:", int(audit_sample["price_to_sma_200d"].isna().sum()))


Dropped all-empty training feature columns:
    feature                                    empty_in
bbb_oas_pit final_train,wf_2018,wf_2019,wf_2020,wf_2021
wf_2018 train_rows= 96200 val_rows= 25306 train_dates= 206 val_dates= 53 purge_cutoff= 2017-12-12
wf_2019 train_rows= 121021 val_rows= 25143 train_dates= 258 val_dates= 52 purge_cutoff= 2018-12-12
wf_2020 train_rows= 146143 val_rows= 25373 train_dates= 310 val_dates= 52 purge_cutoff= 2019-12-12
wf_2021 train_rows= 171501 val_rows= 25674 train_dates= 362 val_dates= 52 purge_cutoff= 2020-12-12
Regime thresholds:
high_vol          0.010106
deep_drawdown    -0.027068
strong_rebound    0.033199
Final train rows: (197658, 102) through 2021-12-13
Test scoring rows: (25791, 91)
Feature count: 85
No-risk rank head feature count: 43
No-risk rank head dropped risk-feature count: 42
No-risk rank head kept features sample: ['return_1d', 'return_5d', 'return_10d', 'return_20d', 'return_60d', 'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentu

In [10]:
# Diagnostics and model training helpers.

RANK_PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "eval_at": [5, 10, 25],
    "label_gain": list(range(N_RANK_BINS)),
    "learning_rate": 0.025,
    "num_leaves": 31,
    "min_child_samples": 80,
    "feature_fraction": 0.72,
    "bagging_fraction": 0.78,
    "bagging_freq": 1,
    "lambda_l1": 0.30,
    "lambda_l2": 1.20,
    "max_depth": 6,
    "verbose": -1,
}
MAG_PARAMS = {**RANK_PARAMS, "objective": "huber", "metric": "l1", "alpha": 0.85}
TAIL_PARAMS = {**RANK_PARAMS, "objective": "binary", "metric": "binary_logloss", "max_depth": 5}


def order_for_lambdarank(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[int]]:
    ordered = frame.sort_values(["date", "ticker"]).reset_index(drop=True)
    groups = ordered.groupby("date", sort=False).size().astype(int).tolist()
    return ordered, groups


def train_rank_model(train_frame: pd.DataFrame, val_frame: pd.DataFrame | None, seed: int, num_boost_round: int | None = None, feature_cols: list[str] | None = None):
    cols = FEATURE_COLS if feature_cols is None else feature_cols
    params = {**RANK_PARAMS, "seed": seed, "feature_fraction_seed": seed, "bagging_seed": seed}
    ordered_train, train_groups = order_for_lambdarank(train_frame)
    train_data = lgb.Dataset(ordered_train[cols], label=ordered_train["alpha_rank_label"].astype(int), group=train_groups)
    if val_frame is None:
        rounds = int(num_boost_round or 80)
        return lgb.train(params, train_data, num_boost_round=rounds), rounds
    ordered_val, val_groups = order_for_lambdarank(val_frame)
    val_data = lgb.Dataset(ordered_val[cols], label=ordered_val["alpha_rank_label"].astype(int), group=val_groups, reference=train_data)
    model = lgb.train(
        params,
        train_data,
        num_boost_round=300,
        valid_sets=[train_data, val_data],
        valid_names=["train", "valid"],
        callbacks=[lgb.early_stopping(60, first_metric_only=True), lgb.log_evaluation(0)],
    )
    return model, int(model.best_iteration or 80)


def train_regression_model(params_base: dict, label_col: str, train_frame: pd.DataFrame, val_frame: pd.DataFrame | None, seed: int, num_boost_round: int | None = None):
    params = {**params_base, "seed": seed, "feature_fraction_seed": seed, "bagging_seed": seed}
    train_data = lgb.Dataset(train_frame[FEATURE_COLS], label=train_frame[label_col].astype(float))
    if val_frame is None:
        rounds = int(num_boost_round or 100)
        return lgb.train(params, train_data, num_boost_round=rounds), rounds
    val_data = lgb.Dataset(val_frame[FEATURE_COLS], label=val_frame[label_col].astype(float), reference=train_data)
    model = lgb.train(
        params,
        train_data,
        num_boost_round=300,
        valid_sets=[train_data, val_data],
        valid_names=["train", "valid"],
        callbacks=[lgb.early_stopping(60, first_metric_only=True), lgb.log_evaluation(0)],
    )
    return model, int(model.best_iteration or 100)


def _datewise_rank(values: pd.Series, dates: pd.Series) -> pd.Series:
    return values.groupby(dates).rank(pct=True)


def score_with_raw_components(frame: pd.DataFrame, rank_raw, mag_raw, tail_raw) -> pd.DataFrame:
    scored = frame[["date", "signal_date", "ticker", "sector"]].copy()
    rank_raw = np.asarray(rank_raw, dtype=float)
    mag_raw = np.asarray(mag_raw, dtype=float)
    tail_raw = np.asarray(tail_raw, dtype=float)
    scored["rank_model_score"] = rank_raw
    scored["magnitude_model_score"] = mag_raw
    scored["tail_risk"] = np.clip(tail_raw, 0.0, 1.0)
    scored["rank_component"] = _datewise_rank(pd.Series(rank_raw, index=frame.index), frame["date"]).to_numpy(float)
    scored["magnitude_component"] = _datewise_rank(pd.Series(mag_raw, index=frame.index), frame["date"]).to_numpy(float)
    scored["tail_component"] = _datewise_rank(pd.Series(tail_raw, index=frame.index), frame["date"]).to_numpy(float)
    scored["model_score"] = (
        RANK_COMPONENT_WEIGHT * scored["rank_component"]
        + MAGNITUDE_COMPONENT_WEIGHT * scored["magnitude_component"]
        + TAIL_COMPONENT_WEIGHT * scored["tail_component"]
    )
    scored["signal_rank"] = scored.groupby("date")["model_score"].rank(pct=True)
    scored["expected_return"] = scored["signal_rank"]
    scored["horizon"] = HORIZON
    scored["expected_volatility"] = frame["vol_20d"].to_numpy(float)
    scored["beta_60d_spy"] = frame["beta_60d_spy"].to_numpy(float)
    scored["regime"] = frame["regime"].to_numpy()
    scored["stress_flag"] = frame["stress_flag"].to_numpy(float)
    return scored.replace([np.inf, -np.inf], np.nan)


def score_with_models(rank_models, magnitude_models, tail_models, frame: pd.DataFrame, rank_feature_cols: list[str] | None = None) -> pd.DataFrame:
    rank_cols = FEATURE_COLS if rank_feature_cols is None else rank_feature_cols
    rank_raw = np.mean([model.predict(frame[rank_cols], num_iteration=model.best_iteration) for model in rank_models], axis=0)
    mag_raw = np.mean([model.predict(frame[FEATURE_COLS], num_iteration=model.best_iteration) for model in magnitude_models], axis=0)
    tail_raw = np.mean([model.predict(frame[FEATURE_COLS], num_iteration=model.best_iteration) for model in tail_models], axis=0)
    return score_with_raw_components(frame, rank_raw, mag_raw, tail_raw)


def _spearman_by_date(frame: pd.DataFrame, score_col: str, target_col: str) -> pd.Series:
    rows = {}
    for date_value, group in frame.groupby("date", sort=True):
        if group[score_col].nunique(dropna=True) < 2 or group[target_col].nunique(dropna=True) < 2:
            rows[date_value] = np.nan
        else:
            rows[date_value] = group[score_col].corr(group[target_col], method="spearman")
    return pd.Series(rows)


def score_diagnostics(frame: pd.DataFrame, score_col: str = "model_score", target_col: str = TARGET_COL, group_col: str | None = None) -> pd.DataFrame:
    work = frame.dropna(subset=[score_col, target_col]).copy()
    if group_col is None:
        work["_group"] = "all"
        group_col = "_group"
    rows = []
    for group_value, group in work.groupby(group_col, sort=True):
        rank_ic = _spearman_by_date(group, score_col, target_col)
        spreads = []
        ndcg25 = []
        for _, g in group.groupby("date", sort=True):
            n = max(1, int(math.ceil(0.20 * len(g))))
            ranked = g.sort_values(score_col)
            spreads.append(ranked.tail(n)[target_col].mean() - ranked.head(n)[target_col].mean())
            y = g["alpha_rank_label"].astype(float).to_numpy()
            s = g[score_col].to_numpy()
            order = np.argsort(s)[::-1][: min(25, len(s))]
            ideal = np.argsort(y)[::-1][: min(25, len(y))]
            discounts = 1.0 / np.log2(np.arange(2, len(order) + 2))
            ndcg25.append(float(np.sum(y[order] * discounts) / np.sum(y[ideal] * discounts)) if np.sum(y[ideal] * discounts) > 0 else np.nan)
        rows.append({
            "group": group_value,
            "dates": int(group["date"].nunique()),
            "mean_rank_ic": float(rank_ic.mean(skipna=True)),
            "std_rank_ic": float(rank_ic.std(skipna=True)),
            "rank_ic_ir": float(rank_ic.mean(skipna=True) / rank_ic.std(skipna=True)) if rank_ic.std(skipna=True) > 0 else np.nan,
            "mean_top_bottom_spread": float(np.nanmean(spreads)),
            "mean_ndcg25": float(np.nanmean(ndcg25)),
        })
    return pd.DataFrame(rows)


def beta_dependence_diagnostics(frame: pd.DataFrame, score_col: str = "model_score") -> pd.DataFrame:
    rows = []
    for date_value, group in frame.dropna(subset=[score_col, "beta_60d_spy"]).groupby("date", sort=True):
        n = max(1, int(math.ceil(0.20 * len(group))))
        top = group.sort_values(score_col, ascending=False).head(n)
        rows.append({
            "date": date_value,
            "top_beta": top["beta_60d_spy"].mean(),
            "universe_beta": group["beta_60d_spy"].mean(),
            "score_beta_corr": group[score_col].corr(group["beta_60d_spy"], method="spearman"),
        })
    result = pd.DataFrame(rows)
    return pd.DataFrame([{
        "dates": len(result),
        "mean_top_beta_minus_universe": (result["top_beta"] - result["universe_beta"]).mean(),
        "mean_score_beta_corr": result["score_beta_corr"].mean(),
    }])


def residualize_model_score_by_date(prediction_frame: pd.DataFrame, feature_frame: pd.DataFrame) -> pd.DataFrame:
    neutralizer_candidates = ["vol_20d", "vol_60d", "cs_rank_vol_60d", "cs_z_vol_60d", "beta_60d_spy", "cs_rank_beta_60d_spy"]
    neutralizer_cols = [col for col in neutralizer_candidates if col in feature_frame.columns]
    extra = feature_frame[["date", "ticker", *neutralizer_cols]].copy()
    work = prediction_frame.merge(extra, on=["date", "ticker"], how="left", suffixes=("", "_feature"))
    rows = []
    for date_value, group in work.groupby("date", sort=True):
        y = group["model_score"].astype(float).to_numpy()
        x_parts = [pd.Series(1.0, index=group.index, name="intercept")]
        for col in neutralizer_cols:
            values = pd.to_numeric(group[col], errors="coerce")
            values = values.fillna(values.median())
            std = values.std(ddof=0)
            if np.isfinite(std) and std > 1e-12:
                x_parts.append(((values - values.mean()) / std).rename(col))
        sector_dummies = pd.get_dummies(group["sector"].fillna("Unknown"), prefix="sector", dtype=float)
        if not sector_dummies.empty:
            x_parts.append(sector_dummies)
        x = pd.concat(x_parts, axis=1).to_numpy(dtype=float)
        fitted = x @ np.linalg.lstsq(x, y, rcond=None)[0]
        residual = y - fitted
        out = group[prediction_frame.columns].copy()
        out["model_score"] = residual
        out["signal_rank"] = pd.Series(residual, index=out.index).rank(pct=True).to_numpy(float)
        out["expected_return"] = out["signal_rank"]
        rows.append(out)
    result = pd.concat(rows, ignore_index=True)
    return result.replace([np.inf, -np.inf], np.nan)


In [11]:
# Ridge baseline alpha selection and multi-seed walk-forward CV.
#
# Task 2 question: if ridge replaces the LightGBM rank head, does the
# resulting portfolio still lose to the nonlinear rank head after costs?
#
# Why the old implementation failed: fold-level ridge IC compared rankers
# but never tested whether any LightGBM edge survived portfolio construction.
#
# Approaches considered:
# 1. Train one final ridge model on final_train. Mirrors the final LightGBM
#    flow and avoids averaging stale fold models.
# 2. Average fold-trained ridge models. More ensemble-like but mismatches
#    the final training window.
# 3. Build linear rank, magnitude, and tail heads. Tests the whole nonlinear
#    stack, but expands scope beyond the rank-head question.
#
# Chosen approach: train one final ridge model and replace only the rank
# head, keeping LightGBM magnitude and tail heads constant. This isolates
# nonlinear rank ordering.
#
# What could break this: if ridge scores have near-zero cross-sectional
# dispersion, the optimizer collapses to constraints and the ablation is
# informative but degenerate.

def ridge_pipeline(alpha: float) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, random_state=2029)),
    ])


def ridge_score(train_frame: pd.DataFrame, val_frame: pd.DataFrame, alpha: float) -> pd.DataFrame:
    model = ridge_pipeline(alpha)
    model.fit(train_frame[FEATURE_COLS], train_frame[TARGET_COL].astype(float))
    out = val_frame[["date", "signal_date", "ticker", "sector", TARGET_COL, "alpha_rank_label", "beta_60d_spy"]].copy()
    out["ridge_score"] = model.predict(val_frame[FEATURE_COLS])
    return out


first_fold = folds[0]
alpha_rows = []
for alpha in RIDGE_ALPHA_GRID:
    scored = ridge_score(first_fold["train"], first_fold["val"], alpha)
    diag = score_diagnostics(scored, score_col="ridge_score").iloc[0]
    alpha_rows.append({"alpha": alpha, "mean_rank_ic": diag["mean_rank_ic"]})
ridge_alpha_selection = pd.DataFrame(alpha_rows).sort_values("mean_rank_ic", ascending=False)
RIDGE_ALPHA = float(ridge_alpha_selection.iloc[0]["alpha"])
print("Ridge alpha selection on first fold:")
print(ridge_alpha_selection.to_string(index=False))
print("Locked RIDGE_ALPHA:", RIDGE_ALPHA)

ridge_rows = []
cv_seed_rows = []
validation_prediction_parts = []
rank_best_iterations = []
mag_best_iterations = []
tail_best_iterations = []

for fold in folds:
    print("\nFold", fold["name"])
    ridge_val = ridge_score(fold["train"], fold["val"], RIDGE_ALPHA)
    ridge_diag = score_diagnostics(ridge_val, score_col="ridge_score").iloc[0].to_dict()
    ridge_rows.append({"fold": fold["name"], **{f"ridge_{k}": v for k, v in ridge_diag.items() if k != "group"}})

    seed_predictions = []
    for seed in ENSEMBLE_SEEDS:
        rank_model, rank_iter = train_rank_model(fold["train"], fold["val"], seed=seed)
        mag_model, mag_iter = train_regression_model(MAG_PARAMS, "alpha_zscore", fold["train"], fold["val"], seed=seed)
        tail_model, tail_iter = train_regression_model(TAIL_PARAMS, "tail_loss_label", fold["train"], fold["val"], seed=seed)
        scored = score_with_models([rank_model], [mag_model], [tail_model], fold["val"])
        scored = scored.merge(
            fold["val"][["date", "ticker", TARGET_COL, "alpha_rank_label", "alpha_zscore", "beta_60d_spy"]],
            on=["date", "ticker"],
            how="left",
            suffixes=("", "_target"),
        )
        diag = score_diagnostics(scored).iloc[0].to_dict()
        cv_seed_rows.append({
            "fold": fold["name"],
            "seed": seed,
            "best_rank_iter": rank_iter,
            "best_mag_iter": mag_iter,
            "best_tail_iter": tail_iter,
            **{k: v for k, v in diag.items() if k != "group"},
        })
        rank_best_iterations.append(rank_iter)
        mag_best_iterations.append(mag_iter)
        tail_best_iterations.append(tail_iter)
        seed_predictions.append(scored)

    combined = (
        pd.concat(seed_predictions, ignore_index=True)
        .groupby(["date", "signal_date", "ticker", "sector"], as_index=False)
        .agg({
            "model_score": "mean",
            "rank_model_score": "mean",
            "magnitude_model_score": "mean",
            "tail_risk": "mean",
            "expected_return": "mean",
            "expected_volatility": "mean",
            "beta_60d_spy": "mean",
            "stress_flag": "mean",
            TARGET_COL: "first",
            "alpha_rank_label": "first",
            "alpha_zscore": "first",
            "regime": "first",
        })
    )
    validation_prediction_parts.append(combined)

ridge_cv_results = pd.DataFrame(ridge_rows)
cv_seed_results = pd.DataFrame(cv_seed_rows)
cv_results = (
    cv_seed_results.groupby("fold")
    .agg(
        best_rank_iter_mean=("best_rank_iter", "mean"),
        best_rank_iter_std=("best_rank_iter", "std"),
        mean_rank_ic_mean=("mean_rank_ic", "mean"),
        mean_rank_ic_std=("mean_rank_ic", "std"),
        std_rank_ic_mean=("std_rank_ic", "mean"),
        rank_ic_ir_mean=("rank_ic_ir", "mean"),
        mean_ndcg25_mean=("mean_ndcg25", "mean"),
        mean_top_bottom_spread_mean=("mean_top_bottom_spread", "mean"),
    )
    .reset_index()
    .merge(ridge_cv_results, on="fold", how="left")
)
oos_validation_predictions = pd.concat(validation_prediction_parts, ignore_index=True)
lgbm_minus_ridge = cv_results["mean_rank_ic_mean"].mean() - cv_results["ridge_mean_rank_ic"].mean()

print("\nRidge vs LightGBM walk-forward diagnostics:")
print(cv_results.to_string(index=False))
print("\nAverage LightGBM rank IC minus Ridge rank IC:", lgbm_minus_ridge)
print("Note: LightGBM uses native NaN handling; Ridge uses median imputation plus standard scaling.")
if lgbm_minus_ridge < 0.01:
    print("Diagnostic note: LightGBM rank IC does not beat ridge by at least 0.01 across folds.")
print("\nValidation beta-dependence diagnostics:")
print(beta_dependence_diagnostics(oos_validation_predictions).to_string(index=False))


Ridge alpha selection on first fold:
 alpha  mean_rank_ic
 100.0      0.007208
  10.0      0.005485
   1.0      0.004977
   0.1      0.004896
Locked RIDGE_ALPHA: 100.0

Fold wf_2018
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[9]	train's ndcg@5: 0.717208	train's ndcg@10: 0.663033	train's ndcg@25: 0.607621	valid's ndcg@5: 0.55064	valid's ndcg@10: 0.531496	valid's ndcg@25: 0.522206
Evaluated only: ndcg@5
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[18]	train's l1: 0.688578	valid's l1: 0.701237
Evaluated only: l1
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[66]	train's binary_logloss: 0.47986	valid's binary_logloss: 0.488851
Evaluated only: binary_logloss
Training until validation scores don't improve for 60 rounds
Early stopping, best iteration is:
[1]	train's ndcg@5: 0.518128	train's ndcg@10: 0.520668	train's ndcg@25: 0.523792	valid'

In [12]:
# Local backtest utilities and cvxpy portfolio optimizer with inverse-vol fallback.
#
# Task 3 question: what happens when the ML model is removed from both
# selection and weighting?
#
# Why the old implementation failed: use_model_alpha=False replaced the
# optimizer alpha vector but still selected names by model_score, so the ML
# model was still driving the candidate universe.
#
# Approaches considered:
# 1. Add a selection_signal field to PortfolioConfig. Smallest explicit API
#    change and keeps all portfolio variants in one path.
# 2. Inject a selector function into the builder. More flexible but heavier
#    than this notebook needs.
# 3. Branch outside the builder. Easy, but risks inconsistent constraints.
#
# Chosen approach: add selection_signal with values model_score or
# inverse_vol. Inverse-vol uses expected_volatility because that is the
# portfolio-facing volatility signal passed through the prediction schema.
#
# What could break this: if expected_volatility is mostly NaN, inverse-vol
# selection degenerates. The ablation cell prints NaN counts before running.

FALLBACK_COUNT = 0
FALLBACK_TESTED = False
OPTIMIZER_FALLBACK_LOG = []


def metrics_from_returns(returns: pd.Series, benchmark_returns: pd.Series | None = None) -> dict[str, float]:
    returns = returns.dropna().astype(float)
    if returns.empty:
        return {}
    nav = (1.0 + returns).cumprod()
    years = max((returns.index.max() - returns.index.min()).days / 365.25, 1 / 252)
    total = float(nav.iloc[-1] / nav.iloc[0] - 1.0)
    annual_return = float((1.0 + total) ** (1.0 / years) - 1.0)
    vol = float(returns.std(ddof=0) * math.sqrt(252.0))
    downside_vol = float(returns.clip(upper=0).std(ddof=0) * math.sqrt(252.0))
    metrics = {
        "total_return": total,
        "annual_return": annual_return,
        "annual_volatility": vol,
        "sharpe": annual_return / vol if vol > 0 else np.nan,
        "sortino": annual_return / downside_vol if downside_vol > 0 else np.nan,
        "max_drawdown": float((nav / nav.cummax() - 1.0).min()),
    }
    if benchmark_returns is not None:
        bench = metrics_from_returns(benchmark_returns)
        metrics["benchmark_annual_return"] = bench["annual_return"]
        metrics["benchmark_sharpe"] = bench["sharpe"]
        metrics["benchmark_max_drawdown"] = bench["max_drawdown"]
        metrics["sharpe_vs_benchmark"] = metrics["sharpe"] - bench["sharpe"]
    return metrics


def approximate_backtest(weights: pd.DataFrame, prices_frame: pd.DataFrame, benchmark: str = BENCHMARK) -> dict[str, object]:
    wide = prices_frame.pivot(index="date", columns="ticker", values="adj_close").sort_index()
    returns = wide.pct_change(fill_method=None).fillna(0.0)
    tickers = [c for c in weights.columns if c in returns.columns]
    aligned = weights.reindex(returns.index).ffill().fillna(0.0).reindex(columns=tickers, fill_value=0.0)
    aligned = aligned.div(aligned.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    turnover = aligned.diff().abs().sum(axis=1) / 2.0
    if not turnover.empty:
        turnover.iloc[0] = aligned.iloc[0].abs().sum()
    strategy_returns = (aligned.shift(1).fillna(0.0) * returns.reindex(columns=tickers).fillna(0.0)).sum(axis=1)
    strategy_returns = strategy_returns - turnover.reindex(strategy_returns.index).fillna(0.0) * (COST_BPS / 10000.0)
    active = strategy_returns.loc[strategy_returns.index >= weights.index.min()]
    bench = returns[benchmark].reindex(active.index).fillna(0.0) if benchmark in returns else None
    metrics = metrics_from_returns(active, bench)
    metrics["average_turnover"] = float(turnover.loc[turnover.index >= weights.index.min()].mean())
    return {"metrics": metrics, "returns": active, "benchmark_returns": bench, "turnover": turnover, "aligned_weights": aligned}


@dataclass
class PortfolioConfig:
    top_fraction: float = 0.28
    min_holdings: int = 80
    max_holdings: int = 160
    max_weight: float = 0.025
    sector_cap: float = 0.24
    beta_target: float = 1.00
    beta_ceiling_normal: float = 1.15
    beta_ceiling_stress: float = 0.95
    lambda_beta: float = 0.50
    lambda_vol: float = 5.0
    lambda_tail: float = 0.35
    lambda_turnover: float = 0.08
    use_model_alpha: bool = True
    selection_signal: str = "model_score"


def _normalize_cap(raw: pd.Series, max_weight: float, sector: pd.Series, sector_cap: float, beta: pd.Series, beta_ceiling: float) -> pd.Series:
    w = raw.clip(lower=0.0).fillna(0.0)
    if w.sum() <= 0:
        w[:] = 1.0
    w = w / w.sum()
    effective_cap = max(max_weight, 1.0 / len(w))
    for _ in range(100):
        old = w.copy()
        w = w.clip(upper=effective_cap)
        for s, names in sector.groupby(sector).groups.items():
            total = w.loc[list(names)].sum()
            if total > sector_cap:
                w.loc[list(names)] *= sector_cap / total
        beta_now = float((w * beta).sum())
        if beta_now > beta_ceiling:
            high = beta > beta.median()
            w.loc[high] *= 0.95
        if w.sum() > 0:
            w = w / w.sum()
        if np.max(np.abs(w - old)) < 1e-8:
            break
    return w / w.sum()


def inverse_vol_fallback(candidates: pd.DataFrame, config: PortfolioConfig, beta_ceiling: float) -> pd.Series:
    raw = 1.0 / candidates["expected_volatility"].replace(0.0, np.nan).fillna(candidates["expected_volatility"].median()).clip(lower=0.005)
    return _normalize_cap(
        pd.Series(raw.to_numpy(), index=candidates["ticker"]),
        config.max_weight,
        candidates.set_index("ticker")["sector"],
        config.sector_cap,
        candidates.set_index("ticker")["beta_60d_spy"].fillna(1.0),
        beta_ceiling,
    )


def build_optimizer_portfolio(predictions: pd.DataFrame, config: PortfolioConfig, strategy_name: str) -> PortfolioWeights:
    global FALLBACK_COUNT, OPTIMIZER_FALLBACK_LOG
    all_tickers = sorted(get_dataset_spec(DATASET_NAME, repo_root=repo_root).tickers)
    rows = []
    index = []
    previous = pd.Series(0.0, index=all_tickers)
    for date_value, frame in predictions.groupby("date", sort=True):
        f = frame.copy()
        stress = bool(f["stress_flag"].fillna(0.0).mean() >= 0.5)
        beta_ceiling = config.beta_ceiling_stress if stress else config.beta_ceiling_normal
        target_n = min(max(config.min_holdings, int(round(len(f) * config.top_fraction))), config.max_holdings, len(f))
        if config.selection_signal == "model_score":
            f["_selection_score"] = f["model_score"]
        elif config.selection_signal == "inverse_vol":
            f["_selection_score"] = 1.0 / f["expected_volatility"].replace(0.0, np.nan).fillna(f["expected_volatility"].median()).clip(lower=0.005)
        else:
            raise ValueError(f"Unknown selection_signal: {config.selection_signal}")
        candidates = f.sort_values("_selection_score", ascending=False).head(target_n).copy()
        tickers = candidates["ticker"].tolist()
        prev = previous.reindex(tickers).fillna(0.0).to_numpy()
        alpha = candidates["model_score"].rank(pct=True).to_numpy(dtype=float)
        if not config.use_model_alpha:
            alpha = (1.0 / candidates["expected_volatility"].clip(lower=0.005)).rank(pct=True).to_numpy(dtype=float)
        alpha = alpha - np.nanmean(alpha)
        beta = candidates["beta_60d_spy"].fillna(1.0).clip(-0.5, 3.0).to_numpy(dtype=float)
        tail = candidates["tail_risk"].fillna(0.2).clip(0.0, 1.0).to_numpy(dtype=float)
        signal_date = pd.Timestamp(candidates["signal_date"].iloc[0])
        hist = returns_wide.loc[returns_wide.index <= signal_date, tickers].tail(60).fillna(0.0)
        sigma = hist.cov().to_numpy(dtype=float)
        sigma = np.nan_to_num((sigma + sigma.T) / 2.0)
        sigma = sigma + np.eye(len(tickers)) * 1e-5
        w = cp.Variable(len(tickers))
        sector_series = candidates.set_index("ticker")["sector"]
        constraints = [w >= 0, cp.sum(w) == 1, w <= config.max_weight, beta @ w <= beta_ceiling]
        for sector_name, names in sector_series.groupby(sector_series).groups.items():
            idx = [tickers.index(name) for name in names if name in tickers]
            if idx:
                constraints.append(cp.sum(w[idx]) <= config.sector_cap)
        objective = cp.Maximize(
            alpha @ w
            - config.lambda_beta * cp.square(beta @ w - config.beta_target)
            - config.lambda_vol * cp.quad_form(w, cp.psd_wrap(sigma))
            - config.lambda_tail * (tail @ w)
            - config.lambda_turnover * cp.norm1(w - prev)
        )
        problem = cp.Problem(objective, constraints)
        solved = False
        weights_selected = None
        for solver in ["CLARABEL", "OSQP", "SCS"]:
            try:
                problem.solve(solver=solver, verbose=False)
            except Exception:
                continue
            if problem.status in {"optimal", "optimal_inaccurate"} and w.value is not None and np.isfinite(w.value).all() and np.sum(w.value) > 0:
                weights_selected = pd.Series(np.asarray(w.value).reshape(-1), index=tickers).clip(lower=0.0)
                weights_selected = weights_selected / weights_selected.sum()
                solved = True
                break
        if not solved:
            FALLBACK_COUNT += 1
            OPTIMIZER_FALLBACK_LOG.append({"strategy_name": strategy_name, "date": pd.Timestamp(date_value), "status": problem.status})
            weights_selected = inverse_vol_fallback(candidates, config, beta_ceiling)
        row = pd.Series(0.0, index=all_tickers)
        row.loc[weights_selected.index] = weights_selected
        rows.append(row / row.sum())
        index.append(pd.Timestamp(date_value))
        previous = row / row.sum()
    weights = pd.DataFrame(rows, index=pd.DatetimeIndex(index))
    weights.index.name = "date"
    weights = validate_weights_frame(weights, dataset_name=DATASET_NAME, repo_root=repo_root)
    return PortfolioWeights(weights=weights, dataset_name=DATASET_NAME, strategy_name=strategy_name, metadata=asdict(config))


def optimizer_fallback_summary() -> pd.DataFrame:
    if not OPTIMIZER_FALLBACK_LOG:
        return pd.DataFrame(columns=["strategy_name", "fallback_count", "first_date", "last_date", "statuses"])
    log = pd.DataFrame(OPTIMIZER_FALLBACK_LOG)
    return (
        log.assign(status=lambda x: x["status"].fillna("unknown").astype(str))
        .groupby("strategy_name", as_index=False)
        .agg(
            fallback_count=("date", "size"),
            first_date=("date", "min"),
            last_date=("date", "max"),
            statuses=("status", lambda s: ",".join(sorted(set(s)))),
        )
        .sort_values(["strategy_name"])
    )


In [13]:
# Fallback path test with deliberately infeasible optimizer settings.

fallback_test_config = PortfolioConfig(max_weight=0.001, min_holdings=200, max_holdings=200, sector_cap=0.05)
fallback_sample = oos_validation_predictions.loc[oos_validation_predictions["date"] == oos_validation_predictions["date"].min()].copy()
fallback_portfolio = build_optimizer_portfolio(fallback_sample, fallback_test_config, "v6_fallback_test")
validate_weights_frame(fallback_portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root)
FALLBACK_TESTED = True
print("Expected infeasible optimizer fallback test passed.")
print("Fallback count so far:", FALLBACK_COUNT)


Expected infeasible optimizer fallback test passed.
Fallback count so far: 1


In [14]:
# Validation-only portfolio sweep.

LOCKED_CONFIG_SELECTION_AUDIT_TEXT = "validation_only_portfolio_sweep_no_test_window_reference"
print(COMPOSITE_SCORE_TUNING_NOTE)
candidate_configs = []
for lambda_beta, lambda_vol, lambda_tail, lambda_turnover, sector_cap, max_weight in itertools.product(
    [0.25, 0.50],
    [2.5, 5.0],
    [0.20, 0.35],
    [0.04, 0.08],
    [0.24],
    [0.025],
):
    candidate_configs.append(PortfolioConfig(
        lambda_beta=lambda_beta,
        lambda_vol=lambda_vol,
        lambda_tail=lambda_tail,
        lambda_turnover=lambda_turnover,
        sector_cap=sector_cap,
        max_weight=max_weight,
    ))

sweep_rows = []
for idx, cfg in enumerate(candidate_configs):
    port = build_optimizer_portfolio(oos_validation_predictions, cfg, f"v6_candidate_{idx}")
    eval_result = approximate_backtest(port.weights, prices)
    metrics = eval_result["metrics"]
    score = metrics["sharpe"] + 0.25 * metrics["annual_return"] - 0.30 * abs(min(metrics["max_drawdown"] + 0.30, 0.0))
    sweep_rows.append({"candidate": idx, "selection_score": score, **metrics, **asdict(cfg)})

portfolio_sweep = pd.DataFrame(sweep_rows).sort_values("selection_score", ascending=False).reset_index(drop=True)
LOCKED_CONFIG = candidate_configs[int(portfolio_sweep.loc[0, "candidate"])]
validation_portfolio = build_optimizer_portfolio(oos_validation_predictions, LOCKED_CONFIG, f"{STRATEGY_NAME}_validation")
validation_eval = approximate_backtest(validation_portfolio.weights, prices)
validation_mc = pd.DataFrame()
MONTE_CARLO_EXECUTED = False
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(2030)
    rows = []
    strategy = validation_eval["returns"].dropna().to_numpy()
    bench = validation_eval["benchmark_returns"].reindex(validation_eval["returns"].dropna().index).fillna(0.0).to_numpy()
    for sim in range(MC_N_SIMS):
        idx = []
        while len(idx) < len(strategy):
            start = int(rng.integers(0, max(1, len(strategy) - MC_BLOCK_SIZE)))
            idx.extend(range(start, min(start + MC_BLOCK_SIZE, len(strategy))))
        idx = idx[: len(strategy)]
        m = metrics_from_returns(pd.Series(strategy[idx]), pd.Series(bench[idx]))
        m["sim"] = sim
        rows.append(m)
    validation_mc = pd.DataFrame(rows)
    MONTE_CARLO_EXECUTED = True

print("Top validation portfolio settings:")
print(portfolio_sweep.head(12).to_string(index=False))
print("\nLocked portfolio config:")
print(pd.Series(asdict(LOCKED_CONFIG)).to_string())
print("\nLocked validation metrics:")
print(pd.Series(validation_eval["metrics"]).to_string())
if MONTE_CARLO_EXECUTED:
    print("\nValidation Monte Carlo summary:")
    print(validation_mc[["annual_return", "sharpe", "max_drawdown", "sharpe_vs_benchmark"]].quantile([0.05, 0.50, 0.95]).to_string())


Composite score weights are held fixed at 0.70/0.25/-0.15. This version tunes optimizer penalties only so signal-blend tuning does not become another validation overfit channel.
Top validation portfolio settings:
 candidate  selection_score  total_return  annual_return  annual_volatility   sharpe  sortino  max_drawdown  benchmark_annual_return  benchmark_sharpe  benchmark_max_drawdown  sharpe_vs_benchmark  average_turnover  top_fraction  min_holdings  max_holdings  max_weight  sector_cap  beta_target  beta_ceiling_normal  beta_ceiling_stress  lambda_beta  lambda_vol  lambda_tail  lambda_turnover  use_model_alpha selection_signal
        15         0.890209      2.929696       0.186713           0.211129 0.884357 1.370889     -0.436090                 0.141208          0.725958               -0.337173             0.158400          0.050294          0.28            80           160       0.025        0.24          1.0                 1.15                 0.95         0.50         5.0    

In [15]:
# Final LightGBM ensemble training.

def robust_rounds(values: list[int], fallback: int, low: int, high: int) -> int:
    clean = [int(v) for v in values if v and np.isfinite(v)]
    if not clean:
        return fallback
    return int(np.clip(round(np.median(clean) + 10), low, high))


def print_iter_distribution(label: str, values: list[int]) -> None:
    clean = pd.Series([int(v) for v in values if v and np.isfinite(v)], dtype=float)
    if clean.empty:
        print(f"{label} best_iter distribution: no valid values")
        return
    q = clean.quantile([0.25, 0.50, 0.75])
    print(
        f"{label} best_iter distribution: "
        f"min={int(clean.min())} "
        f"p25={int(q.loc[0.25])} "
        f"median={int(q.loc[0.50])} "
        f"p75={int(q.loc[0.75])} "
        f"max={int(clean.max())}"
    )


print_iter_distribution("Rank head", rank_best_iterations)
print_iter_distribution("Magnitude head", mag_best_iterations)
print_iter_distribution("Tail head", tail_best_iterations)

FINAL_RANK_ROUNDS = robust_rounds(rank_best_iterations, 80, 25, 180)
FINAL_MAG_ROUNDS = robust_rounds(mag_best_iterations, 100, 25, 220)
FINAL_TAIL_ROUNDS = robust_rounds(tail_best_iterations, 100, 25, 220)
final_rank_models = []
final_no_risk_rank_models = []
final_magnitude_models = []
final_tail_models = []
final_ridge_model = ridge_pipeline(RIDGE_ALPHA)
final_ridge_model.fit(final_train[FEATURE_COLS], final_train[TARGET_COL].astype(float))

print("Final train rows:", len(final_train))
print("Final rank rounds:", FINAL_RANK_ROUNDS)
print("Final magnitude rounds:", FINAL_MAG_ROUNDS)
print("Final tail rounds:", FINAL_TAIL_ROUNDS)
print("Final ridge alpha:", RIDGE_ALPHA)
for seed in ENSEMBLE_SEEDS:
    rank_model, _ = train_rank_model(final_train, None, seed=seed, num_boost_round=FINAL_RANK_ROUNDS)
    no_risk_rank_model, _ = train_rank_model(final_train, None, seed=seed, num_boost_round=FINAL_RANK_ROUNDS, feature_cols=NO_RISK_FEATURE_COLS)
    mag_model, _ = train_regression_model(MAG_PARAMS, "alpha_zscore", final_train, None, seed=seed, num_boost_round=FINAL_MAG_ROUNDS)
    tail_model, _ = train_regression_model(TAIL_PARAMS, "tail_loss_label", final_train, None, seed=seed, num_boost_round=FINAL_TAIL_ROUNDS)
    final_rank_models.append(rank_model)
    final_no_risk_rank_models.append(no_risk_rank_model)
    final_magnitude_models.append(mag_model)
    final_tail_models.append(tail_model)
    print("Trained ensemble seed", seed)

model_bundle = {
    "rank_models": final_rank_models,
    "no_risk_rank_models": final_no_risk_rank_models,
    "magnitude_models": final_magnitude_models,
    "tail_models": final_tail_models,
    "feature_names": FEATURE_COLS,
    "no_risk_feature_names": NO_RISK_FEATURE_COLS,
    "regime_thresholds": regime_thresholds,
    "portfolio_config": asdict(LOCKED_CONFIG),
}


Rank head best_iter distribution: min=1 p25=3 median=8 p75=16 max=83
Magnitude head best_iter distribution: min=2 p25=10 median=19 p75=68 max=192
Tail head best_iter distribution: min=59 p25=68 median=127 p75=207 max=300
Final train rows: 197658
Final rank rounds: 25
Final magnitude rounds: 29
Final tail rounds: 137
Final ridge alpha: 100.0
Trained ensemble seed 17
Trained ensemble seed 42
Trained ensemble seed 101
Trained ensemble seed 211


In [16]:
# Local prediction, portfolio, and stress-proxy evaluation.

predictions = score_with_models(final_rank_models, final_magnitude_models, final_tail_models, test_features)
no_risk_rank_predictions = score_with_models(final_no_risk_rank_models, final_magnitude_models, final_tail_models, test_features, rank_feature_cols=NO_RISK_FEATURE_COLS)
risk_residualized_predictions = residualize_model_score_by_date(predictions, test_features)
ridge_rank_raw = final_ridge_model.predict(test_features[FEATURE_COLS])
lgbm_mag_raw = np.mean([model.predict(test_features[FEATURE_COLS], num_iteration=model.best_iteration) for model in final_magnitude_models], axis=0)
lgbm_tail_raw = np.mean([model.predict(test_features[FEATURE_COLS], num_iteration=model.best_iteration) for model in final_tail_models], axis=0)
ridge_predictions = score_with_raw_components(test_features, ridge_rank_raw, lgbm_mag_raw, lgbm_tail_raw)
ridge_score_dispersion = ridge_predictions.groupby("date")["rank_model_score"].std()
no_risk_score_dispersion = no_risk_rank_predictions.groupby("date")["rank_model_score"].std()
predictions_for_validation = validate_prediction_frame(
    predictions[["date", "ticker", "horizon", "expected_return", "expected_volatility"]],
    dataset_name=DATASET_NAME,
    horizon=HORIZON,
    repo_root=repo_root,
)
scored_test_labeled = predictions.merge(
    test_labeled[["date", "ticker", TARGET_COL, "alpha_rank_label", "alpha_zscore", "beta_60d_spy", "regime"]],
    on=["date", "ticker"],
    how="left",
    suffixes=("", "_target"),
)
if "regime_target" in scored_test_labeled:
    scored_test_labeled["regime"] = scored_test_labeled["regime_target"].fillna(scored_test_labeled["regime"])
    scored_test_labeled = scored_test_labeled.drop(columns=["regime_target"])
residualized_test_labeled = risk_residualized_predictions.merge(
    test_labeled[["date", "ticker", TARGET_COL, "alpha_rank_label", "beta_60d_spy"]],
    on=["date", "ticker"],
    how="left",
    suffixes=("", "_target"),
)
no_risk_test_labeled = no_risk_rank_predictions.merge(
    test_labeled[["date", "ticker", TARGET_COL, "alpha_rank_label", "beta_60d_spy"]],
    on=["date", "ticker"],
    how="left",
    suffixes=("", "_target"),
)

portfolio = build_optimizer_portfolio(predictions, LOCKED_CONFIG, STRATEGY_NAME)
validated_weights = validate_weights_frame(portfolio.weights, dataset_name=DATASET_NAME, repo_root=repo_root)
local_prices = prices.loc[(prices["date"] >= TEST_START - pd.Timedelta(days=90)) & (prices["date"] <= TEST_END)].copy()
local_eval = approximate_backtest(portfolio.weights, local_prices)
robustness_prices = prices.loc[(prices["date"] >= TEST_START - pd.Timedelta(days=90)) & (prices["date"] <= pd.Timestamp("2025-12-31"))].copy()
robustness_eval = approximate_backtest(portfolio.weights, robustness_prices)
robustness_returns = robustness_eval["returns"]
robustness_benchmark = robustness_eval["benchmark_returns"]
robustness_rows = []
for label, start, end in [
    ("2022", pd.Timestamp("2022-01-03"), pd.Timestamp("2022-12-31")),
    ("2023", pd.Timestamp("2023-01-01"), pd.Timestamp("2023-12-31")),
    ("2024", pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")),
    ("2025", pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31")),
    ("2022_2025", pd.Timestamp("2022-01-03"), pd.Timestamp("2025-12-31")),
]:
    mask = (robustness_returns.index >= start) & (robustness_returns.index <= end)
    period_returns = robustness_returns.loc[mask]
    if period_returns.empty:
        continue
    period_benchmark = robustness_benchmark.reindex(period_returns.index).fillna(0.0) if robustness_benchmark is not None else None
    row = {"period": label, **metrics_from_returns(period_returns, period_benchmark)}
    row["average_turnover"] = float(robustness_eval["turnover"].reindex(period_returns.index).mean())
    robustness_rows.append(row)
post_lock_robustness_table = pd.DataFrame(robustness_rows)

print("Predictions:", predictions.shape)
print("Ridge-rank prediction cross-sectional std by date:")
print(ridge_score_dispersion.describe()[["min", "50%", "mean", "max"]].to_string())
print("No-risk rank prediction cross-sectional std by date:")
print(no_risk_score_dispersion.describe()[["min", "50%", "mean", "max"]].to_string())
print("Weights:", validated_weights.shape)
print("Average names held:", float((validated_weights > 0).sum(axis=1).mean()))
print("Max single-name weight:", float(validated_weights.max(axis=1).max()))
print("Test signal metrics:")
print(score_diagnostics(scored_test_labeled).to_string(index=False))
print("Risk-residualized score test signal metrics:")
print(score_diagnostics(residualized_test_labeled).to_string(index=False))
print("No-risk rank-head test signal metrics:")
print(score_diagnostics(no_risk_test_labeled).to_string(index=False))
print("Test beta-dependence diagnostics:")
print(beta_dependence_diagnostics(scored_test_labeled).to_string(index=False))
print("Risk-residualized beta-dependence diagnostics:")
print(beta_dependence_diagnostics(residualized_test_labeled).to_string(index=False))
print("No-risk rank-head beta-dependence diagnostics:")
print(beta_dependence_diagnostics(no_risk_test_labeled).to_string(index=False))
print("\nBrian LightGBM v6 local test window")
print("Test window:", TEST_START.date(), "to", TEST_END.date())
for key, value in local_eval["metrics"].items():
    print(f"{key}: {value}")
print("\nPost-lock robustness report. Not used for config selection.")
print(post_lock_robustness_table[[
    "period", "annual_return", "annual_volatility", "sharpe", "max_drawdown",
    "benchmark_annual_return", "benchmark_sharpe", "benchmark_max_drawdown",
    "sharpe_vs_benchmark", "average_turnover",
]].to_string(index=False))


Predictions: (25791, 18)
Ridge-rank prediction cross-sectional std by date:
min     0.002264
50%     0.003510
mean    0.003688
max     0.006553
No-risk rank prediction cross-sectional std by date:
min     0.037483
50%     0.092160
mean    0.079262
max     0.107897
Weights: (52, 503)
Average names held: 64.5576923076923
Max single-name weight: 0.02499999999429031
Test signal metrics:
group  dates  mean_rank_ic  std_rank_ic  rank_ic_ir  mean_top_bottom_spread  mean_ndcg25
  all     52     -0.008678     0.118996   -0.072927               -0.001824     0.519064
Risk-residualized score test signal metrics:
group  dates  mean_rank_ic  std_rank_ic  rank_ic_ir  mean_top_bottom_spread  mean_ndcg25
  all     52      0.006222     0.093738    0.066375                0.000156     0.512562
No-risk rank-head test signal metrics:
group  dates  mean_rank_ic  std_rank_ic  rank_ic_ir  mean_top_bottom_spread  mean_ndcg25
  all     52     -0.000376     0.089116   -0.004216               -0.000857     0.494

In [17]:
# Ablations, including ML-only, ridge-rank, and true model-unused isolation.

print("Inverse-vol ablation expected_volatility NaNs:", int(predictions["expected_volatility"].isna().sum()))
print("Inverse-vol ablation vol_20d proxy NaNs:", int(test_features["vol_20d"].isna().sum()))

ablation_configs = {
    "full_v6": (predictions, LOCKED_CONFIG),
    "ml_only": (predictions, PortfolioConfig(**{
        **asdict(LOCKED_CONFIG),
        "lambda_beta": 0.0,
        "lambda_vol": 0.0,
        "lambda_tail": 0.0,
        "lambda_turnover": 0.0,
    })),
    "risk_residualized_score": (risk_residualized_predictions, LOCKED_CONFIG),
    "no_risk_rank_head": (no_risk_rank_predictions, LOCKED_CONFIG),
    "ridge_only_portfolio": (ridge_predictions, LOCKED_CONFIG),
    "optimizer_alpha_replaced_with_inverse_vol": (predictions, PortfolioConfig(**{
        **asdict(LOCKED_CONFIG),
        "use_model_alpha": False,
        "selection_signal": "model_score",
    })),
    "model_unused": (predictions, PortfolioConfig(**{
        **asdict(LOCKED_CONFIG),
        "use_model_alpha": False,
        "selection_signal": "inverse_vol",
    })),
    "no_beta_penalty": (predictions, PortfolioConfig(**{**asdict(LOCKED_CONFIG), "lambda_beta": 0.0})),
    "no_tail_penalty": (predictions, PortfolioConfig(**{**asdict(LOCKED_CONFIG), "lambda_tail": 0.0})),
}
ablation_rows = []
ablation_ports = {}
for name, (prediction_frame, cfg) in ablation_configs.items():
    port = build_optimizer_portfolio(prediction_frame, cfg, f"ablation_{name}")
    ablation_ports[name] = port
    metrics = approximate_backtest(port.weights, local_prices)["metrics"]
    row = {"ablation": name, **metrics}
    if name == "ridge_only_portfolio":
        row["ridge_rank_score_std_mean"] = float(ridge_score_dispersion.mean())
        row["ridge_rank_score_std_min"] = float(ridge_score_dispersion.min())
    if name == "no_risk_rank_head":
        row["no_risk_rank_score_std_mean"] = float(no_risk_score_dispersion.mean())
        row["no_risk_rank_score_std_min"] = float(no_risk_score_dispersion.min())
    ablation_rows.append(row)
ablation_table = pd.DataFrame(ablation_rows).sort_values("sharpe", ascending=False)
print("Ablation table using local test proxy:")
print(ablation_table[["ablation", "annual_return", "annual_volatility", "sharpe", "max_drawdown", "average_turnover"]].to_string(index=False))
print("\nRidge-only score dispersion:")
print(ridge_score_dispersion.describe().to_string())
print("\nNo-risk rank-head score dispersion:")
print(no_risk_score_dispersion.describe().to_string())
fallback_summary = optimizer_fallback_summary()
print("\nOptimizer fallback summary:")
if fallback_summary.empty:
    print("No optimizer fallbacks occurred.")
else:
    print(fallback_summary.to_string(index=False))


Inverse-vol ablation expected_volatility NaNs: 6
Inverse-vol ablation vol_20d proxy NaNs: 6
Ablation table using local test proxy:
                                 ablation  annual_return  annual_volatility    sharpe  max_drawdown  average_turnover
                          no_tail_penalty       0.180361           0.230708  0.781773     -0.153545          0.077637
                                  full_v6       0.158170           0.228940  0.690881     -0.152269          0.078164
                          no_beta_penalty       0.102959           0.215940  0.476795     -0.137246          0.071790
                                  ml_only       0.090948           0.223094  0.407668     -0.143957          0.093185
optimizer_alpha_replaced_with_inverse_vol       0.024462           0.178477  0.137058     -0.144881          0.077272
                        no_risk_rank_head       0.018703           0.255980  0.073065     -0.197267          0.059300
                     ridge_only_portfolio  

In [18]:
# Alpha contribution decomposition.
#
# Task 1 question: of the active return relative to a naive top-K
# equal-weight portfolio, how much comes from each weighting layer?
#
# Why the old implementation failed: it compared three portfolios with
# different selected names and weights, so the components could not sum
# cleanly and the residual silently absorbed selection differences.
#
# Approaches considered:
# 1. Hold selected universe constant and vary only weights. This cleanly
#    isolates weighting decisions.
# 2. Hold final weights constant and perturb signals. More model-centric
#    but hard to explain because constraints bind nonlinearly.
# 3. Compare separate named ablation portfolios. Easy, but not additive.
#
# Chosen approach: hold the top-K names by model_score constant on each
# date, where K is the average full-portfolio holdings count, then compare
# equal-weight, inverse-vol, model-rank, and cvxpy weights.
#
# What could break this: if cvxpy falls back on more than about 10% of
# dates, optimizer value is not distinguishable from fallback behavior.
# Those dates are excluded from additive attribution and the fallback rate
# is printed beside the table.

def _weights_frame_from_rows(rows: list[pd.Series]) -> pd.DataFrame:
    weights = pd.DataFrame(rows).reindex(columns=UNIVERSE_TICKERS, fill_value=0.0)
    weights.index.name = "date"
    weights = weights.div(weights.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    return validate_weights_frame(weights, dataset_name=DATASET_NAME, repo_root=repo_root)


def fixed_universe_simple_weights(predictions_frame: pd.DataFrame, k: int, mode: str, config: PortfolioConfig) -> pd.DataFrame:
    rows = []
    for date_value, frame in predictions_frame.groupby("date", sort=True):
        candidates = frame.sort_values("model_score", ascending=False).head(k).copy()
        tickers = candidates["ticker"].tolist()
        if mode == "equal":
            raw = pd.Series(1.0, index=tickers)
            selected = raw / raw.sum()
        elif mode == "inverse_vol":
            raw = 1.0 / candidates.set_index("ticker")["expected_volatility"].replace(0.0, np.nan).fillna(candidates["expected_volatility"].median()).clip(lower=0.005)
            selected = _normalize_cap(
                raw,
                config.max_weight,
                candidates.set_index("ticker")["sector"],
                config.sector_cap,
                candidates.set_index("ticker")["beta_60d_spy"].fillna(1.0),
                config.beta_ceiling_stress if candidates["stress_flag"].fillna(0.0).mean() >= 0.5 else config.beta_ceiling_normal,
            )
        elif mode == "model_rank":
            raw = candidates.set_index("ticker")["model_score"].rank(pct=True).clip(lower=0.01)
            selected = _normalize_cap(
                raw,
                config.max_weight,
                candidates.set_index("ticker")["sector"],
                config.sector_cap,
                candidates.set_index("ticker")["beta_60d_spy"].fillna(1.0),
                config.beta_ceiling_stress if candidates["stress_flag"].fillna(0.0).mean() >= 0.5 else config.beta_ceiling_normal,
            )
        else:
            raise ValueError(f"Unknown fixed-universe mode: {mode}")
        row = pd.Series(0.0, index=UNIVERSE_TICKERS)
        row.loc[selected.index] = selected.to_numpy(float)
        row.name = pd.Timestamp(date_value)
        rows.append(row)
    return _weights_frame_from_rows(rows)


def fixed_universe_cvxpy_weights(predictions_frame: pd.DataFrame, k: int, config: PortfolioConfig) -> tuple[pd.DataFrame, list[pd.Timestamp]]:
    rows = []
    fallback_dates = []
    previous = pd.Series(0.0, index=UNIVERSE_TICKERS)
    for date_value, frame in predictions_frame.groupby("date", sort=True):
        candidates = frame.sort_values("model_score", ascending=False).head(k).copy()
        tickers = candidates["ticker"].tolist()
        stress = bool(candidates["stress_flag"].fillna(0.0).mean() >= 0.5)
        beta_ceiling = config.beta_ceiling_stress if stress else config.beta_ceiling_normal
        prev = previous.reindex(tickers).fillna(0.0).to_numpy()
        alpha = candidates["model_score"].rank(pct=True).to_numpy(dtype=float)
        alpha = alpha - np.nanmean(alpha)
        beta = candidates["beta_60d_spy"].fillna(1.0).clip(-0.5, 3.0).to_numpy(dtype=float)
        tail = candidates["tail_risk"].fillna(0.2).clip(0.0, 1.0).to_numpy(dtype=float)
        signal_date = pd.Timestamp(candidates["signal_date"].iloc[0])
        hist = returns_wide.loc[returns_wide.index <= signal_date, tickers].tail(60).fillna(0.0)
        sigma = hist.cov().to_numpy(dtype=float)
        sigma = np.nan_to_num((sigma + sigma.T) / 2.0)
        sigma = sigma + np.eye(len(tickers)) * 1e-5
        w = cp.Variable(len(tickers))
        sector_series = candidates.set_index("ticker")["sector"]
        constraints = [w >= 0, cp.sum(w) == 1, w <= config.max_weight, beta @ w <= beta_ceiling]
        for sector_name, names in sector_series.groupby(sector_series).groups.items():
            idx = [tickers.index(name) for name in names if name in tickers]
            if idx:
                constraints.append(cp.sum(w[idx]) <= config.sector_cap)
        objective = cp.Maximize(
            alpha @ w
            - config.lambda_beta * cp.square(beta @ w - config.beta_target)
            - config.lambda_vol * cp.quad_form(w, cp.psd_wrap(sigma))
            - config.lambda_tail * (tail @ w)
            - config.lambda_turnover * cp.norm1(w - prev)
        )
        problem = cp.Problem(objective, constraints)
        selected = None
        for solver in ["CLARABEL", "OSQP", "SCS"]:
            try:
                problem.solve(solver=solver, verbose=False)
            except Exception:
                continue
            if problem.status in {"optimal", "optimal_inaccurate"} and w.value is not None and np.isfinite(w.value).all() and np.sum(w.value) > 0:
                selected = pd.Series(np.asarray(w.value).reshape(-1), index=tickers).clip(lower=0.0)
                selected = selected / selected.sum()
                break
        if selected is None:
            fallback_dates.append(pd.Timestamp(date_value))
            OPTIMIZER_FALLBACK_LOG.append({"strategy_name": "attribution_cvxpy_fixed", "date": pd.Timestamp(date_value), "status": problem.status})
            selected = inverse_vol_fallback(candidates, config, beta_ceiling)
        row = pd.Series(0.0, index=UNIVERSE_TICKERS)
        row.loc[selected.index] = selected.to_numpy(float)
        row.name = pd.Timestamp(date_value)
        rows.append(row)
        previous = row / row.sum()
    return _weights_frame_from_rows(rows), fallback_dates


def portfolio_returns_from_weights(weights: pd.DataFrame) -> pd.Series:
    return approximate_backtest(weights, local_prices)["returns"]


def annualized_linear_return(series: pd.Series) -> float:
    return float(series.dropna().mean() * 252.0)


attribution_k = int(round((portfolio.weights > 0).sum(axis=1).mean()))
attribution_k = max(1, min(attribution_k, len(UNIVERSE_TICKERS)))
attribution_equal_weights = fixed_universe_simple_weights(predictions, attribution_k, "equal", LOCKED_CONFIG)
attribution_inverse_vol_weights = fixed_universe_simple_weights(predictions, attribution_k, "inverse_vol", LOCKED_CONFIG)
attribution_model_rank_weights = fixed_universe_simple_weights(predictions, attribution_k, "model_rank", LOCKED_CONFIG)
attribution_cvxpy_weights, attribution_fallback_dates = fixed_universe_cvxpy_weights(predictions, attribution_k, LOCKED_CONFIG)

attribution_returns = {
    "equal_weight": portfolio_returns_from_weights(attribution_equal_weights),
    "inverse_vol": portfolio_returns_from_weights(attribution_inverse_vol_weights),
    "model_rank": portfolio_returns_from_weights(attribution_model_rank_weights),
    "cvxpy_full": portfolio_returns_from_weights(attribution_cvxpy_weights),
}
common_index = attribution_returns["equal_weight"].index
for series in attribution_returns.values():
    common_index = common_index.intersection(series.index)
if attribution_fallback_dates:
    common_index = common_index.difference(pd.DatetimeIndex(attribution_fallback_dates))

eq_ret = attribution_returns["equal_weight"].reindex(common_index).fillna(0.0)
inv_ret = attribution_returns["inverse_vol"].reindex(common_index).fillna(0.0)
rank_ret = attribution_returns["model_rank"].reindex(common_index).fillna(0.0)
cvx_ret = attribution_returns["cvxpy_full"].reindex(common_index).fillna(0.0)

contribution_series = {
    "inverse_vol_weighting_layer": inv_ret - eq_ret,
    "model_rank_weighting_layer": rank_ret - inv_ret,
    "cvxpy_optimizer_layer": cvx_ret - rank_ret,
}
active_total_series = cvx_ret - eq_ret
residual_series = active_total_series.copy()
for series in contribution_series.values():
    residual_series = residual_series - series
contribution_series["residual_component"] = residual_series

total_active_annual_return = annualized_linear_return(active_total_series)
rows = []
for name, series in contribution_series.items():
    ann = annualized_linear_return(series)
    rows.append({
        "component": name,
        "annualized_contribution": ann,
        "pct_of_total_active_annual_return": ann / total_active_annual_return if abs(total_active_annual_return) > 1e-12 else np.nan,
    })
alpha_contribution_table = pd.DataFrame(rows)
alpha_decomposition_residual_pct_abs = abs(annualized_linear_return(residual_series)) / max(abs(total_active_annual_return), 1e-12)
alpha_decomposition_fallback_rate = len(attribution_fallback_dates) / max(1, predictions["date"].nunique())

full_holdings = (portfolio.weights > 0)
fixed_holdings = (attribution_equal_weights > 0).reindex(full_holdings.index).fillna(False)
full_fixed_overlap = (full_holdings & fixed_holdings).sum(axis=1).mean() / max(1, attribution_k)

print("Alpha contribution decomposition:")
print("Selected universe: top-K by model_score with K =", attribution_k)
print("Mean full-vs-attribution selected-name overlap:", float(full_fixed_overlap))
print("Attribution cvxpy fallback rate:", alpha_decomposition_fallback_rate)
if alpha_decomposition_fallback_rate > 0.10:
    print("Diagnostic note: attribution fallback rate exceeds 10%; optimizer contribution is noisy.")
print(alpha_contribution_table.to_string(index=False))
print("Alpha decomposition residual pct abs:", alpha_decomposition_residual_pct_abs)


Alpha contribution decomposition:
Selected universe: top-K by model_score with K = 65
Mean full-vs-attribution selected-name overlap: 0.8863905325443787
Attribution cvxpy fallback rate: 0.0
                  component  annualized_contribution  pct_of_total_active_annual_return
inverse_vol_weighting_layer            -1.569638e-02                      -1.720353e-01
 model_rank_weighting_layer             4.708398e-02                       5.160492e-01
      cvxpy_optimizer_layer             5.985173e-02                       6.559861e-01
         residual_component            -1.251800e-18                      -1.371996e-17
Alpha decomposition residual pct abs: 1.3719960078541467e-17


In [19]:
# Feature importance and submission prediction check.

final_importance_rows = []
for model_name, models, model_features in [
    ("rank", final_rank_models, FEATURE_COLS),
    ("no_risk_rank", final_no_risk_rank_models, NO_RISK_FEATURE_COLS),
    ("magnitude", final_magnitude_models, FEATURE_COLS),
    ("tail", final_tail_models, FEATURE_COLS),
]:
    for model_idx, model in enumerate(models):
        gains = model.feature_importance(importance_type="gain")
        splits = model.feature_importance(importance_type="split")
        for feature, gain, split in zip(model_features, gains, splits):
            final_importance_rows.append({"model": model_name, "model_idx": model_idx, "feature": feature, "gain": float(gain), "split": int(split)})
importance = pd.DataFrame(final_importance_rows).groupby("feature").agg(gain_mean=("gain", "mean"), split_mean=("split", "mean")).sort_values("gain_mean", ascending=False)
print("Top 30 features:")
print(importance.head(30).to_string())


def predict_from_prices(model, prices_frame: pd.DataFrame, dates=None, tickers=None) -> pd.DataFrame:
    feature_frame = build_model_features(prices_frame)
    all_dates = pd.DatetimeIndex(pd.to_datetime(prices_frame["date"].sort_values().unique()).tz_localize(None))
    if dates is None:
        calendar = weekly_first_trading_day_calendar(prices_frame, all_dates.min(), all_dates.max())
    else:
        requested = pd.DatetimeIndex(pd.to_datetime(pd.Series(dates), utc=True).dt.tz_localize(None).sort_values().unique())
        rows = []
        for execution_date in requested:
            pos = all_dates.searchsorted(execution_date, side="left")
            if pos < len(all_dates) and all_dates[pos] == execution_date and pos > 0:
                rows.append({"signal_date": pd.Timestamp(all_dates[pos - 1]), "date": pd.Timestamp(execution_date)})
        calendar = pd.DataFrame(rows)
    scoring = calendar.merge(feature_frame.rename(columns={"date": "signal_date"}), on="signal_date", how="left")
    scoring["sector"] = scoring["ticker"].map(ticker_to_sector)
    scoring = add_regime_columns(scoring, model["regime_thresholds"])
    if tickers is not None:
        scoring = scoring.loc[scoring["ticker"].isin([t.upper() for t in tickers])].copy()
    scored = score_with_models(model["rank_models"], model["magnitude_models"], model["tail_models"], scoring)
    return scored[["date", "ticker", "horizon", "expected_return", "expected_volatility", "signal_date", "model_score", "rank_model_score", "magnitude_model_score", "tail_risk", "beta_60d_spy", "sector", "regime", "stress_flag"]]


submission_check = predict_from_prices(model_bundle, prices, dates=portfolio.weights.index, tickers=UNIVERSE_TICKERS)
print("Submission prediction check:", submission_check.shape)
print(submission_check.head().to_string(index=False))


Top 30 features:
                             gain_mean  split_mean
feature                                           
cs_z_vol_60d              13330.000883   57.833333
cs_rank_vol_60d            6611.945987   48.416667
beta_60d_ff_mktrf          2291.314839   66.666667
dgs2_pit                   1540.985947   84.416667
dgs10_pit                  1155.730959   65.583333
beta_60d_ff_smb            1145.425502   57.750000
cs_z_vol_20d               1138.554269   20.833333
cs_rank_vol_20d            1103.400967   28.250000
yield_10y_2y_spread_pit    1087.495172   59.583333
beta_60d_ff_hml            1039.269402   65.000000
beta_60d_ff_rmw            1022.408403   52.083333
beta_60d_spy                755.816064   37.166667
vol_60d                     752.831492   36.416667
cs_rank_beta_60d_spy        728.052881   30.416667
usd_index_pit               722.130601   57.125000
spy_vol_20d                 711.389120   44.000000
intraday_range              663.843529   35.000000
cs_z_momentum_

In [25]:
# Optional artifact saving and MLflow logging.

artifact_dir = repo_root / "MODELS" / "Brian" / "v6_artifacts"
saved_model_paths = []
metadata = {
    "model_name": MODEL_NAME,
    "model_version": "v6",
    "dataset": DATASET_NAME,
    "benchmark": BENCHMARK,
    "horizon": HORIZON,
    "target": TARGET_COL,
    "feature_count": len(FEATURE_COLS),
    "features": FEATURE_COLS,
    "no_risk_rank_feature_count": len(NO_RISK_FEATURE_COLS),
    "no_risk_rank_features": NO_RISK_FEATURE_COLS,
    "no_risk_rank_dropped_features": NO_RISK_DROPPED_FEATURES,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "ridge_alpha": RIDGE_ALPHA,
    "cv_results": cv_results.to_dict(orient="records"),
    "ridge_cv_results": ridge_cv_results.to_dict(orient="records"),
    "portfolio_config": asdict(LOCKED_CONFIG),
    "external_data_sources": EXTERNAL_DATA_SOURCES,
    "backtest_metrics": local_eval["metrics"],
    "post_lock_robustness_table": post_lock_robustness_table.to_dict(orient="records"),
    "ablation_table": ablation_table.to_dict(orient="records"),
    "alpha_contribution_table": alpha_contribution_table.to_dict(orient="records"),
    "notes": "V6 audits LightGBM against ridge and isolates alpha contribution using an explicit market-neutralized sector residual target and cvxpy portfolio optimizer.",
}

if SAVE_LOCAL_ARTIFACTS:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    for prefix, models in [("rank", final_rank_models), ("no_risk_rank", final_no_risk_rank_models), ("magnitude", final_magnitude_models), ("tail", final_tail_models)]:
        for seed, model_obj in zip(ENSEMBLE_SEEDS, models):
            path = artifact_dir / f"lgbm_v6_{prefix}_seed_{seed}.txt"
            model_obj.save_model(str(path))
            saved_model_paths.append(path)
    metadata_path = artifact_dir / "lgbm_v6_metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    saved_model_paths.append(metadata_path)
    print("Saved local artifacts to", artifact_dir)
else:
    metadata_path = None
    print("SAVE_LOCAL_ARTIFACTS is False. No model files were written.")

if RUN_MLFLOW:
    import mlflow
    init_mlflow(repo_root=repo_root)
    with start_run(run_name=MODEL_NAME, dataset_name=DATASET_NAME, tags={"model_type": "lightgbm", "version": "6", "strategy_type": "market_sector_residual_alpha_audit"}, repo_root=repo_root):
        mlflow.log_params({
            "model_name": MODEL_NAME,
            "dataset": DATASET_NAME,
            "horizon": HORIZON,
            "feature_count": len(FEATURE_COLS),
            "ridge_alpha": RIDGE_ALPHA,
            "rank_rounds": FINAL_RANK_ROUNDS,
            "magnitude_rounds": FINAL_MAG_ROUNDS,
            "tail_rounds": FINAL_TAIL_ROUNDS,
            "submission_mode": SUBMISSION_MODE,
        })
        log_predictions(predictions_for_validation)
        log_portfolio(portfolio)
        if SAVE_LOCAL_ARTIFACTS and saved_model_paths:
            log_model_submission(
                {path.stem: path for path in saved_model_paths},
                model_name=MODEL_NAME,
                model_family="lightgbm",
                feature_names=FEATURE_COLS,
                target=TARGET_COL,
                horizon=HORIZON,
                rebalance_frequency="weekly_first_trading_day",
                preprocessing={"missing_values": "native_lightgbm", "ridge_scaler": "median_impute_standard_scale"},
                model_config=metadata,
                source_files=[repo_root / "MODELS" / "Brian" / "brian_lgbm_v6.ipynb"],
                notes=metadata["notes"],
            )
else:
    print("RUN_MLFLOW is False. Set True only when intentionally logging.")


SAVE_LOCAL_ARTIFACTS is False. No model files were written.
RUN_MLFLOW is False. Set True only when intentionally logging.


In [21]:
# Final self-check cell.

assert "ridge_cv_results" in globals() and len(ridge_cv_results) > 0, "Ridge baseline rank IC was not computed."
assert "ml_only" in set(ablation_table["ablation"]), "ML-only ablation row missing."
assert "ridge_only_portfolio" in set(ablation_table["ablation"]), "Ridge-only portfolio ablation row missing."
assert "model_unused" in set(ablation_table["ablation"]), "Model-unused ablation row missing."
assert "risk_residualized_score" in set(ablation_table["ablation"]), "Risk-residualized score ablation row missing."
assert "no_risk_rank_head" in set(ablation_table["ablation"]), "No-risk rank-head ablation row missing."
assert "optimizer_alpha_replaced_with_inverse_vol" in set(ablation_table["ablation"]), "Renamed inverse-vol optimizer-alpha ablation row missing."
assert "alpha_contribution_table" in globals() and len(alpha_contribution_table) > 0, "Alpha contribution decomposition missing."
assert "post_lock_robustness_table" in globals() and len(post_lock_robustness_table) >= 2, "Post-lock robustness report missing."
assert alpha_decomposition_residual_pct_abs < 0.05, "Alpha contribution residual is too large; decomposition is not clean."
assert LEAKAGE_AUDIT_SAMPLE_SIZE >= 25, "Leakage audit sample size must be at least 25."
assert "specialist_rank_models" not in model_bundle, "Regime-specialized model objects must not exist in model_bundle."
assert EMBARGO_DAYS >= 2 * HORIZON, "Embargo must be at least 2 * horizon."
assert LEAKAGE_AUDIT_PASSED is True, "Leakage audit did not pass."
assert FALLBACK_TESTED is True, "Optimizer fallback test did not run."
assert "2022" not in LOCKED_CONFIG_SELECTION_AUDIT_TEXT, "Locked config selection cell references test window."
if SUBMISSION_MODE:
    assert MONTE_CARLO_EXECUTED is True, "Submission mode requires Monte Carlo execution."

print("V6 self-check passed.")
print("Ridge baseline rows:", len(ridge_cv_results))
print("Ablation rows:", ablation_table["ablation"].tolist())
print("Fallback count:", FALLBACK_COUNT, "(includes the deliberate infeasible fallback test)")


V6 self-check passed.
Ridge baseline rows: 4
Ablation rows: ['no_tail_penalty', 'full_v6', 'no_beta_penalty', 'ml_only', 'optimizer_alpha_replaced_with_inverse_vol', 'no_risk_rank_head', 'ridge_only_portfolio', 'model_unused', 'risk_residualized_score']
Fallback count: 3 (includes the deliberate infeasible fallback test)
